# 第9章 混合モデルとEMアルゴリズム (Mixture Models and EM) — 演習問題完全解答・数値検証

本ノートブックは、C.M. Bishop 著『Pattern Recognition and Machine Learning (PRML)』**第9章「混合モデルとEMアルゴリズム (Mixture Models and EM)」**に収録されている **全27問（Exercises 9.1 〜 9.27）** の完全な論理証明、数式ステップ穴埋め問題、および自己完結した Python 数値検証コードを網羅した演習ノートブックです。

---

## 目次 (Table of Contents)

1. **K-means、基本GMMおよびパラメータ事前分布 (Exercises 9.1 〜 9.9)**
   - [Exercise 9.1: K-means アルゴリズムの有限回反復収束証明](#Exercise-9.1)
   - [Exercise 9.2: Robbins-Monro 逐次推定法によるオンライン K-means 更新則の導出](#Exercise-9.2)
   - [Exercise 9.3: ガウス混合モデルにおける周辺分布と事後負担率（ベイズの定理）の導出](#Exercise-9.3)
   - [Exercise 9.4: パラメータ事前分布を持つモデルにおける MAP 推定 EM アルゴリズム](#Exercise-9.4)
   - [Exercise 9.5: d分離による潜在変数事後分布のデータ点間因数分解証明](#Exercise-9.5)
   - [Exercise 9.6: 共通共分散行列 $\mathbf{\Sigma}_k = \mathbf{\Sigma}$ を持つ GMM の EM アルゴリズム導出](#Exercise-9.6)
   - [Exercise 9.7: 完全データ対数尤度最大化による各成分パラメータの独立最尤推定](#Exercise-9.7)
   - [Exercise 9.8: 期待完全データ対数尤度の中心ベクトル $\boldsymbol{\mu}_k$ に関する閉形式最大化](#Exercise-9.8)
   - [Exercise 9.9: 期待完全データ対数尤度の共分散 $\mathbf{\Sigma}_k$ および混合係数 $\pi_k$ に関する閉形式最大化](#Exercise-9.9)

2. **一般混合モデル、ベルヌーイ混合モデル (BMM) (Exercises 9.10 〜 9.18)**
   - [Exercise 9.10: 分割ベクトルにおける条件付き分布の混合分布表現](#Exercise-9.10)
   - [Exercise 9.11: 分散ゼロ極限 $\epsilon \to 0$ における EM アルゴリズムと K-means の等価性](#Exercise-9.11)
   - [Exercise 9.12: 一般混合分布全体の平均および共分散行列の合成則 (式 9.49, 9.50)](#Exercise-9.12)
   - [Exercise 9.13: ベルヌーイ混合モデルにおける同一初期値での1反復退化現象の証明](#Exercise-9.13)
   - [Exercise 9.14: ベルヌーイ混合モデルの同時分布と潜在変数周辺化による観測分布の導出](#Exercise-9.14)
   - [Exercise 9.15: ベルヌーイ混合モデルにおける中心パラメータ $\boldsymbol{\mu}_k$ の M ステップ再推定式 (9.59) の導出](#Exercise-9.15)
   - [Exercise 9.16: ベルヌーイ混合モデルにおける混合係数 $\pi_k$ の M ステップ更新式 (9.60) の導出](#Exercise-9.16)
   - [Exercise 9.17: ベルヌーイ混合モデルにおける対数尤度の上界性と特異点（発散）の非存在証明](#Exercise-9.17)
   - [Exercise 9.18: Beta-Dirichlet 事前分布を持つベルヌーイ混合モデルの MAP-EM アルゴリズム導出](#Exercise-9.18)

3. **多項分布混合、回帰EM、変分下界の幾何、インクリメンタルEM (Exercises 9.19 〜 9.27)**
   - [Exercise 9.19: 多項分布混合モデル (Mixture of Multinomials) の EM アルゴリズム導出](#Exercise-9.19)
   - [Exercise 9.20: ベイズ線形回帰における超パラメータ $\alpha$ の EM アルゴリズム再推定式 (9.63) の導出](#Exercise-9.20)
   - [Exercise 9.21: ベイズ線形回帰におけるノイズ精度 $\beta$ の EM アルゴリズム再推定式の導出](#Exercise-9.21)
   - [Exercise 9.22: 関連ベクトルマシン (RVM) における超パラメータ再推定式 (9.67, 9.68) の EM 導出](#Exercise-9.22)
   - [Exercise 9.23: RVM における周辺尤度直接最大化と EM アルゴリズムの形式的等価性証明](#Exercise-9.23)
   - [Exercise 9.24: 対数尤度の変分分解 $\ln p(\mathbf{X}|\boldsymbol{\theta}) = \mathcal{L}(q, \boldsymbol{\theta}) + \mathrm{KL}(q \| p)$ の証明](#Exercise-9.24)
   - [Exercise 9.25: 接点における変分下界と対数尤度の勾配一致定理の証明](#Exercise-9.25)
   - [Exercise 9.26: オンライン（インクリメンタル）EM における中心ベクトル逐次更新式 (9.78, 9.79) の導出](#Exercise-9.26)
   - [Exercise 9.27: インクリメンタル EM における共分散行列 $\mathbf{\Sigma}_k$ および混合係数 $\pi_k$ の逐次更新式の導出](#Exercise-9.27)

---


In [1]:
# 共通ライブラリのインポートとパス設定
import sys, os
sys.path.append(os.path.abspath('../'))
import numpy as np
import matplotlib.pyplot as plt

np.set_printoptions(precision=4, suppress=True)
print("Environment successfully initialized for Chapter 9 exercises.")

Environment successfully initialized for Chapter 9 exercises.


---
## <a id="Exercise-9.1 ⭐"></a>Exercise 9.1 ⭐: K-means アルゴリズムの有限回反復収束証明

### 問題の提示
9.1節で論じられた K-means アルゴリズムを考える。$N$ 個のデータ点を $K$ 個のクラスタに割り当てる場合の数が高々 $K^N$ 通り（有限集合）であること、および歪み尺度
$$ J = \sum_{n=1}^N \sum_{k=1}^K r_{nk} \|\mathbf{x}_n - \boldsymbol{\mu}_k\|^2 \quad (9.1) $$
が各ステップで単調非増加であることを用いて、K-means アルゴリズムが必ず有限回の反復で厳密に停止（収束）することを示せ。

### [解答の道筋と穴埋め]
1. **有限状態空間**:
   $N$ 点の各々に $1$ から $K$ のクラスタラベルを割り当てる全配置の集合は、高々 [ ① ] 通りの有限集合である。
2. **中心ベクトルの最適性**:
   クラスタ割り当て $\{r_{nk}\}$ が固定されたとき、歪み尺度 $J$ は各 $\boldsymbol{\mu}_k$ に関して凸な2次形式であり、
   $$ \boldsymbol{\mu}_k = \frac{\sum_n r_{nk} \mathbf{x}_n}{\sum_n r_{nk}} \quad (9.4) $$
   において大域的かつ [ ② ] に最小化される。
3. **歪み尺度の厳密な単調減少性**:
   - $r_{nk}$ の更新ステップ（割り当てステップ）：各点を最も近い中心 $\boldsymbol{\mu}_k$ に割り当てるため、$J$ は非増加（減少または不変）となる。
   - $\boldsymbol{\mu}_k$ の更新ステップ（中心更新ステップ）：重心を再計算するため、$J$ は非増加となる。
   割り当て $\{r_{nk}\}$ に変化が生じる限り、$J$ は前回の値より厳密に減少する。
4. **有限ステップ停止**:
   $J$ が厳密に減少するため、アルゴリズムが過去に訪れた割り当て配置に再び戻る（サイクルする）ことはあり得ない。
   可能な割り当ての総数は有限であるため、高々有限回のステップで割り当てが一切変化しない状態（$\Delta r_{nk} = 0$）に到達し、アルゴリズムは確実に停止する。

### 穴埋めの解答
- ①: $K^N$
- ②: 一意

In [2]:
# Exercise 9.1 数値検証: K-means の有限ステップ停止性と歪み尺度 J の単調非増加性
import numpy as np

np.random.seed(42)
N, D, K = 60, 2, 3
X = np.vstack([
    np.random.randn(20, D) + np.array([0, 0]),
    np.random.randn(20, D) + np.array([4, 4]),
    np.random.randn(20, D) + np.array([-4, 4]),
])

# 初期クラスタ中心
centers = X[np.random.choice(N, K, replace=False)].copy()
J_history = []
assignment_history = []

for step in range(100):
    # 1. 割り当てステップ (r_nk)
    dists = np.linalg.norm(X[:, None, :] - centers[None, :, :], axis=2)**2
    assignments = np.argmin(dists, axis=1)
    J = np.sum(np.min(dists, axis=1))
    J_history.append(J)
    assignment_history.append(tuple(assignments))

    # 収束判定
    if step > 0 and assignment_history[-1] == assignment_history[-2]:
        print(f"K-means terminated strictly at step {step} with {len(set(assignment_history))} visited states.")
        break

    # 2. 中心更新ステップ
    for k in range(K):
        pts = X[assignments == k]
        if len(pts) > 0:
            centers[k] = np.mean(pts, axis=0)

# J が単調減少していることを検証
diffs = np.diff(J_history)
assert np.all(diffs <= 1e-10), "Distortion J must be monotonically non-increasing!"
print(f"Exercise 9.1 verified: J decreased from {J_history[0]:.2f} to {J_history[-1]:.2f} without cycling.")

K-means terminated strictly at step 3 with 3 visited states.
Exercise 9.1 verified: J decreased from 670.20 to 99.76 without cycling.


---
## <a id="Exercise-9.2"></a>Exercise 9.2: Robbins-Monro 逐次推定法によるオンライン K-means 更新則の導出

### 問題の提示
2.3.5節で論じられた Robbins-Monro 逐次根探索手順を、回帰関数
$$ g(\boldsymbol{\mu}_k) = \mathbb{E}_{\mathbf{x}} [2 r_{nk} (\boldsymbol{\mu}_k - \mathbf{x})] $$
の根（すなわち歪み尺度 $J$ の勾配期待値がゼロとなる点）の探索に適用せよ。
これにより、逐次 K-means 更新式 (9.5)
$$ \boldsymbol{\mu}_k^{(\tau)} = \boldsymbol{\mu}_k^{(\tau-1)} + \eta_\tau (\mathbf{x}_n - \boldsymbol{\mu}_k^{(\tau-1)}) \quad (9.5) $$
（ここでデータ点 $\mathbf{x}_n$ がクラスタ $k$ に割り当てられた場合）が自然に導出されることを示せ。

### [解答の道筋と穴埋め]
1. **Robbins-Monro の一般形式**:
   目的とする回帰関数 $g(\boldsymbol{\mu}) = 0$ の根を求めるため、観測された確率的サンプル $z_\tau$ を用いて：
   $$ \boldsymbol{\mu}^{(\tau)} = \boldsymbol{\mu}^{(\tau-1)} - a_\tau [ \text{①} ] $$
   の形で更新を行う。ここで学習率 $a_\tau$ は $\sum_\tau a_\tau = \infty, \sum_\tau a_\tau^2 < \infty$ を満たす。
2. **K-means 歪み関数の勾配観測**:
   単一データ点 $\mathbf{x}_n$ に対する歪み $J_n = \sum_k r_{nk} \|\mathbf{x}_n - \boldsymbol{\mu}_k\|^2$ の $\boldsymbol{\mu}_k$ に関する勾配は：
   $$ \nabla_{\boldsymbol{\mu}_k} J_n = 2 r_{nk} (\boldsymbol{\mu}_k - \mathbf{x}_n) $$
   データ点 $\mathbf{x}_n$ がクラスタ $k$ に割り当てられている（$r_{nk} = 1$）とき、不偏勾配観測値は $z_\tau = 2(\boldsymbol{\mu}_k^{(\tau-1)} - \mathbf{x}_n)$ となる。
3. **逐次更新式の整合**:
   これを Robbins-Monro 更新則に代入すると：
   $$ \boldsymbol{\mu}_k^{(\tau)} = \boldsymbol{\mu}_k^{(\tau-1)} - a_\tau \cdot 2(\boldsymbol{\mu}_k^{(\tau-1)} - \mathbf{x}_n) = \boldsymbol{\mu}_k^{(\tau-1)} + [ \text{②} ] (\mathbf{x}_n - \boldsymbol{\mu}_k^{(\tau-1)}) $$
   学習率 $\eta_\tau = 2 a_\tau$ と置くことで、式 (9.5) が厳密に得られる。

### 穴埋めの解答
- ①: $z_\tau$ (勾配の観測サンプル)
- ②: $2 a_\tau$

In [3]:
# Exercise 9.2 数値検証: Robbins-Monro 逐次 K-means と標本平均の漸近一致
N = 1000
true_mean = np.array([3.0, -2.0])
X_samples = true_mean + np.random.randn(N, 2)

# 逐次推定 (Robbins-Monro: eta_tau = 1 / tau)
mu_seq = np.array([0.0, 0.0])
for tau in range(1, N + 1):
    x_n = X_samples[tau - 1]
    eta_tau = 1.0 / tau
    mu_seq = mu_seq + eta_tau * (x_n - mu_seq)

batch_mean = np.mean(X_samples, axis=0)
np.testing.assert_allclose(mu_seq, batch_mean, atol=1e-12)
print(f"Exercise 9.2 verified: Robbins-Monro sequential estimate {mu_seq} exactly equals batch mean {batch_mean}!")

Exercise 9.2 verified: Robbins-Monro sequential estimate [ 3.0378 -1.9628] exactly equals batch mean [ 3.0378 -1.9628]!


---
## <a id="Exercise-9.3"></a>Exercise 9.3: ガウス混合モデルにおける周辺分布と事後負担率（ベイズの定理）の導出

### 問題の提示
潜在変数 $\mathbf{z} \in \{0, 1\}^K$（1-of-K 表現）の周辺分布が式 (9.10)
$$ p(\mathbf{z}) = \prod_{k=1}^K \pi_k^{z_k} \quad (9.10) $$
で与えられ、$\mathbf{z}$ が与えられたときの観測変数 $\mathbf{x}$ の条件付き分布が式 (9.11)
$$ p(\mathbf{x}|\mathbf{z}) = \prod_{k=1}^K \mathcal{N}(\mathbf{x}|\boldsymbol{\mu}_k, \mathbf{\Sigma}_k)^{z_k} \quad (9.11) $$
で与えられるガウス混合モデル (GMM) を考える。
1. $\mathbf{z}$ を周辺化することにより、周辺分布 $p(\mathbf{x})$ が式 (9.7) の標準的な混合ガウス分布になることを示せ。
2. ベイズの定理を用いて、事後負担率 $\gamma(z_k) \equiv p(z_k=1|\mathbf{x})$ が式 (9.13) で与えられることを示せ。

### [解答の道筋と穴埋め]
1. **潜在変数の周辺化**:
   $p(\mathbf{x}) = \sum_{\mathbf{z}} p(\mathbf{z}) p(\mathbf{x}|\mathbf{z})$ を計算する。
   $\mathbf{z}$ の取り得る状態は、$j$ 番目の要素のみが 1 で他が 0 である $K$ 個の単位ベクトル $\mathbf{e}_j$ である。
   $\mathbf{z} = \mathbf{e}_j$ のとき、$p(\mathbf{z}) = \pi_j$ かつ $p(\mathbf{x}|\mathbf{z}) = \mathcal{N}(\mathbf{x}|\boldsymbol{\mu}_j, \mathbf{\Sigma}_j)$ となるため：
   $$ p(\mathbf{x}) = \sum_{j=1}^K \pi_j [ \text{①} ] $$
2. **事後確率（負担率）の導出**:
   ベイズの定理より：
   $$ \gamma(z_k) = p(z_k = 1 | \mathbf{x}) = \frac{p(z_k = 1) p(\mathbf{x} | z_k = 1)}{\sum_{j=1}^K p(z_j = 1) p(\mathbf{x} | z_j = 1)} = \frac{\pi_k \mathcal{N}(\mathbf{x}|\boldsymbol{\mu}_k, \mathbf{\Sigma}_k)}{\sum_{j=1}^K \pi_j \mathcal{N}(\mathbf{x}|\boldsymbol{\mu}_j, \mathbf{\Sigma}_j)} = [ \text{②} ] $$
   これにより、Eステップで計算される負担率の確率論的意味が明確になる。

### 穴埋めの解答
- ①: $\mathcal{N}(\mathbf{x}|\boldsymbol{\mu}_j, \mathbf{\Sigma}_j)$
- ②: $\gamma(z_k)$

In [4]:
# Exercise 9.3 数値検証: 1-of-K 潜在変数周辺化とベイズ負担率の厳密性
K = 3
pi = np.array([0.2, 0.5, 0.3])
mu = np.array([-2.0, 0.0, 3.0])
sigma = np.array([0.5, 1.0, 0.8])

def norm_pdf(x, m, s):
    return np.exp(-0.5 * ((x - m) / s)**2) / (np.sqrt(2 * np.pi) * s)

# テスト点
x_test = 0.5

# 1. 周辺分布 p(x)
p_x = sum(pi[k] * norm_pdf(x_test, mu[k], sigma[k]) for k in range(K))

# 2. 事後負担率 gamma(z_k)
gamma = np.array([pi[k] * norm_pdf(x_test, mu[k], sigma[k]) for k in range(K)]) / p_x
np.testing.assert_allclose(np.sum(gamma), 1.0, atol=1e-12)

# 周辺密度の数値積分による規格化確認
grid = np.linspace(-10, 10, 1000)
p_grid = sum(pi[k] * norm_pdf(grid, mu[k], sigma[k]) for k in range(K))
integral = np.trapezoid(p_grid, grid)
np.testing.assert_allclose(integral, 1.0, atol=1e-3)

print(f"Exercise 9.3 verified: Marginal p(x={x_test}) = {p_x:.4f}, Responsibilities = {gamma}")

Exercise 9.3 verified: Marginal p(x=0.5) = 0.1772, Responsibilities = [0.     0.9936 0.0064]


---
## <a id="Exercise-9.4"></a>Exercise 9.4: パラメータ事前分布を持つモデルにおける MAP 推定 EM アルゴリズム

### 問題の提示
観測データ $\mathbf{X}$ と潜在変数 $\mathbf{Z}$ を含む確率モデルにおいて、パラメータ $\boldsymbol{\theta}$ に対する事前分布 $p(\boldsymbol{\theta})$ が与えられているとする。
パラメータの事後確率 $p(\boldsymbol{\theta}|\mathbf{X})$ を最大化（MAP 推定）するために EM アルゴリズムを適用する場合：
1. Eステップは最尤推定の場合と全く同一であること
2. Mステップにおいて最大化すべき目的関数が
   $$ \mathcal{Q}(\boldsymbol{\theta}, \boldsymbol{\theta}^{(\mathrm{old})}) + \ln p(\boldsymbol{\theta}) $$
   となることを示せ。

### [解答の道筋と穴埋め]
1. **パラメータ事後分布の対数分解**:
   $$ \ln p(\boldsymbol{\theta}|\mathbf{X}) = \ln p(\mathbf{X}, \boldsymbol{\theta}) - \ln p(\mathbf{X}) = \ln p(\mathbf{X}|\boldsymbol{\theta}) + \ln p(\boldsymbol{\theta}) - \ln p(\mathbf{X}) $$
   ここで $\ln p(\mathbf{X})$ は $\boldsymbol{\theta}$ に依存しない定数である。
2. **対数尤度の変分下界の導入**:
   任意の潜在変数分布 $q(\mathbf{Z})$ に対し：
   $$ \ln p(\mathbf{X}|\boldsymbol{\theta}) = \mathcal{L}(q, \boldsymbol{\theta}) + \mathrm{KL}(q \| p(\mathbf{Z}|\mathbf{X}, \boldsymbol{\theta})) $$
   両辺に $\ln p(\boldsymbol{\theta})$ を加えると：
   $$ \ln p(\mathbf{X}, \boldsymbol{\theta}) = \mathcal{L}(q, \boldsymbol{\theta}) + \ln p(\boldsymbol{\theta}) + \mathrm{KL}(q \| p(\mathbf{Z}|\mathbf{X}, \boldsymbol{\theta})) $$
3. **Eステップ**:
   現在のパラメータ $\boldsymbol{\theta}^{(\mathrm{old})}$ において下界を最大化するには、$\mathrm{KL}$ ダイバージェンスをゼロにすればよい。
   $\boldsymbol{\theta}$ の事前分布は $\mathbf{Z}$ に依存しないため、最適な $q(\mathbf{Z})$ は：
   $$ q(\mathbf{Z}) = [ \text{①} ] $$
   となり、最尤推定の場合と完全に同一である。
4. **Mステップ**:
   固定された $q(\mathbf{Z}) = p(\mathbf{Z}|\mathbf{X}, \boldsymbol{\theta}^{(\mathrm{old})})$ の下で、下界 $\mathcal{L}(q, \boldsymbol{\theta}) + \ln p(\boldsymbol{\theta})$ を $\boldsymbol{\theta}$ に関して最大化する。
   期待完全データ対数尤度 $\mathcal{Q}(\boldsymbol{\theta}, \boldsymbol{\theta}^{(\mathrm{old})}) = \mathbb{E}_{\mathbf{Z}}[\ln p(\mathbf{X}, \mathbf{Z}|\boldsymbol{\theta})]$ を用いると、
   最大化対象は：
   $$ [ \text{②} ] $$
   となる。

### 穴埋めの解答
- ①: $p(\mathbf{Z}|\mathbf{X}, \boldsymbol{\theta}^{(\mathrm{old})})$
- ②: $\mathcal{Q}(\boldsymbol{\theta}, \boldsymbol{\theta}^{(\mathrm{old})}) + \ln p(\boldsymbol{\theta})$

In [5]:
# Exercise 9.4 数値検証: MAP-EM における Eステップの同一性と Mステップ下界単調増加
# 1次元2成分 GMM with Gaussian prior on means: p(mu_k) = N(mu_k | 0, sigma0^2)
np.random.seed(42)
N = 50
X = np.concatenate([np.random.randn(25) * 0.5 - 2, np.random.randn(25) * 0.5 + 2])
sigma0_sq = 4.0 # Prior variance on mu

# 初期値
mu = np.array([-0.5, 0.5])
pi = np.array([0.5, 0.5])
var = np.array([1.0, 1.0])

def compute_map_objective(X, mu, pi, var, sigma0_sq):
    # ln p(X | theta) + ln p(theta)
    log_lik = 0.0
    for x in X:
        p_x = sum(pi[k] * np.exp(-0.5 * (x - mu[k])**2 / var[k]) / np.sqrt(2 * np.pi * var[k]) for k in range(2))
        log_lik += np.log(p_x)
    log_prior = -0.5 * np.sum(mu**2) / sigma0_sq
    return log_lik + log_prior

map_history = [compute_map_objective(X, mu, pi, var, sigma0_sq)]

for it in range(10):
    # Eステップ: 最尤と全く同一
    resp = np.zeros((N, 2))
    for n, x in enumerate(X):
        dens = np.array([pi[k] * np.exp(-0.5 * (x - mu[k])**2 / var[k]) / np.sqrt(2 * np.pi * var[k]) for k in range(2)])
        resp[n] = dens / np.sum(dens)

    # Mステップ: MAP 更新 (mu_k の解)
    # d/d mu_k [ -0.5 * sum_n gamma_{nk} (x_n - mu_k)^2 / var_k - 0.5 * mu_k^2 / sigma0_sq ] = 0
    # => mu_k * (N_k / var_k + 1 / sigma0_sq) = sum_n gamma_{nk} x_n / var_k
    N_k = resp.sum(axis=0)
    for k in range(2):
        mu[k] = (np.sum(resp[:, k] * X) / var[k]) / (N_k[k] / var[k] + 1.0 / sigma0_sq)
    pi = N_k / N
    
    map_history.append(compute_map_objective(X, mu, pi, var, sigma0_sq))

diffs = np.diff(map_history)
assert np.all(diffs >= -1e-10), "MAP objective must monotonically increase!"
print(f"Exercise 9.4 verified: MAP objective monotonically increased from {map_history[0]:.2f} to {map_history[-1]:.2f}")

Exercise 9.4 verified: MAP objective monotonically increased from -133.10 to -86.79


---
## <a id="Exercise-9.5"></a>Exercise 9.5: d分離による潜在変数事後分布のデータ点間因数分解証明

### 問題の提示
図 9.6 に示されるガウス混合モデルの有向グラフを考える。
8.2節で論じられた d分離基準（d-separation criterion）を用いて、所与のパラメータ $(\boldsymbol{\mu}, \mathbf{\Sigma}, \boldsymbol{\pi})$ および全観測データ $\mathbf{X}$ の下で、潜在変数 $\mathbf{Z} = \{\mathbf{z}_1, \dots, \mathbf{z}_N\}$ の事後分布が各データ点について完全に因数分解すること：
$$ p(\mathbf{Z}|\mathbf{X}, \boldsymbol{\mu}, \mathbf{\Sigma}, \boldsymbol{\pi}) = \prod_{n=1}^N p(\mathbf{z}_n | \mathbf{x}_n, \boldsymbol{\mu}, \mathbf{\Sigma}, \boldsymbol{\pi}) \quad (9.28) $$
を示せ。

### [解答の道筋と穴埋め]
1. **有向グラフの構造**:
   パラメータノード $\{\boldsymbol{\mu}, \mathbf{\Sigma}, \boldsymbol{\pi}\}$ から各潜在変数 $\mathbf{z}_n$ へエッジが伸び、各潜在変数から対応する観測変数 $\mathbf{x}_n$ へエッジ $\mathbf{z}_n \rightarrow \mathbf{x}_n$ が伸びている。
2. **異なるデータ点間の経路**:
   任意の異なる2つの潜在変数 $\mathbf{z}_m$ と $\mathbf{z}_n$ ($m \ne n$) の間の唯一の経路は、共通の親である [ ① ] ノードを経由する分岐経路（tail-to-tail）
   $$ \mathbf{z}_m \leftarrow \{\boldsymbol{\mu}, \mathbf{\Sigma}, \boldsymbol{\pi}\} \rightarrow \mathbf{z}_n $$
   である。
3. **d分離の判定**:
   パラメータ $\{\boldsymbol{\mu}, \mathbf{\Sigma}, \boldsymbol{\pi}\}$ は [ ② ] されているため、分岐経路（tail-to-tail）は完全に遮断（ブロック）される。
   また、観測ノード $\mathbf{x}_m, \mathbf{x}_n$ は合流点（head-to-head）を持たない。
   したがって：
   $$ \mathbf{z}_m \perp\!\!\!\perp \mathbf{z}_n \mid \mathbf{X}, \boldsymbol{\mu}, \mathbf{\Sigma}, \boldsymbol{\pi} $$
   が任意の $m \ne n$ について成立し、全潜在変数の事後確率は各データ点ごとの積へと因数分解する。

### 穴埋めの解答
- ①: パラメータ
- ②: 観測（所与と）

In [6]:
# Exercise 9.5 数値検証: 2点 GMM における結合事後分布と独立積の完全一致
from prml.graphical import check_d_separation

# 1. d分離のグラフ検証
# adj: 'params' -> 'z1', 'params' -> 'z2', 'z1' -> 'x1', 'z2' -> 'x2'
adj = {
    'params': ['z1', 'z2'],
    'z1': ['x1'],
    'z2': ['x2'],
    'x1': [],
    'x2': []
}
# params と x1, x2 を所与としたとき、z1 と z2 は d分離される
is_separated = check_d_separation(adj, ['z1'], ['z2'], ['params', 'x1', 'x2'])
assert is_separated, "z1 and z2 must be d-separated given params and observations!"

# 2. 確率テンソルの直接計算
pi = np.array([0.4, 0.6])
mu = np.array([1.0, -1.0])
x1, x2 = 0.8, -1.2

# p(z1, z2 | x1, x2)
joint_p = np.zeros((2, 2))
for z1 in range(2):
    for z2 in range(2):
        joint_p[z1, z2] = (pi[z1] * np.exp(-0.5 * (x1 - mu[z1])**2)) * (pi[z2] * np.exp(-0.5 * (x2 - mu[z2])**2))
joint_p /= joint_p.sum()

# 各点独立の p(z1 | x1) * p(z2 | x2)
p_z1 = np.array([pi[k] * np.exp(-0.5 * (x1 - mu[k])**2) for k in range(2)])
p_z1 /= p_z1.sum()
p_z2 = np.array([pi[k] * np.exp(-0.5 * (x2 - mu[k])**2) for k in range(2)])
p_z2 /= p_z2.sum()

np.testing.assert_allclose(joint_p, np.outer(p_z1, p_z2), atol=1e-12)
print("Exercise 9.5 verified: p(z1, z2 | x1, x2, theta) strictly factorizes into p(z1|x1) * p(z2|x2)!")

Exercise 9.5 verified: p(z1, z2 | x1, x2, theta) strictly factorizes into p(z1|x1) * p(z2|x2)!


---
## <a id="Exercise-9.6"></a>Exercise 9.6: 共通共分散行列 $\mathbf{\Sigma}_k = \mathbf{\Sigma}$ を持つ GMM の EM アルゴリズム導出

### 問題の提示
すべての混合成分が共通の共分散行列 $\mathbf{\Sigma}_k = \mathbf{\Sigma}$ を共有するガウス混合モデルを考える。
この制約の下で尤度関数を最大化する EM アルゴリズムの更新方程式を導出せよ。

### [解答の道筋と穴埋め]
1. **完全データ期待対数尤度 $\mathcal{Q}$**:
   $$ \mathcal{Q} = -\frac{1}{2} \sum_{n=1}^N \sum_{k=1}^K \gamma(z_{nk}) \left[ D \ln(2\pi) + \ln|\mathbf{\Sigma}| + (\mathbf{x}_n - \boldsymbol{\mu}_k)^{\mathrm{T}} \mathbf{\Sigma}^{-1} (\mathbf{x}_n - \boldsymbol{\mu}_k) \right] + \sum_{n=1}^N \sum_{k=1}^K \gamma(z_{nk}) \ln \pi_k $$
2. **中心ベクトル $\boldsymbol{\mu}_k$ および混合係数 $\pi_k$ の更新**:
   $\boldsymbol{\mu}_k$ および $\pi_k$ に関する項は共通共分散の場合も通常モデルと全く同一の形をしているため、
   $$ \boldsymbol{\mu}_k = \frac{1}{N_k} \sum_{n=1}^N \gamma(z_{nk}) \mathbf{x}_n, \quad \pi_k = [ \text{①} ] $$
   となる。ここで $N_k = \sum_{n=1}^N \gamma(z_{nk})$。
3. **共通共分散行列 $\mathbf{\Sigma}$ の最大化**:
   トレースの性質 $\mathbf{v}^{\mathrm{T}} \mathbf{\Sigma}^{-1} \mathbf{v} = \mathrm{Tr}(\mathbf{\Sigma}^{-1} \mathbf{v} \mathbf{v}^{\mathrm{T}})$ を用いて $\mathbf{\Sigma}^{-1}$ で微分する：
   $$ \frac{\partial \mathcal{Q}}{\partial \mathbf{\Sigma}^{-1}} = \frac{1}{2} \left[ \sum_{n=1}^N \sum_{k=1}^K \gamma(z_{nk}) \right] \mathbf{\Sigma} - \frac{1}{2} \sum_{n=1}^N \sum_{k=1}^K \gamma(z_{nk}) (\mathbf{x}_n - \boldsymbol{\mu}_k)(\mathbf{x}_n - \boldsymbol{\mu}_k)^{\mathrm{T}} $$
   $\sum_{k=1}^K \gamma(z_{nk}) = 1$ より、第1項の係数は $\sum_{n=1}^N 1 = N$ となる。これをゼロと置くことで：
   $$ \mathbf{\Sigma} = [ \text{②} ] \sum_{k=1}^K \sum_{n=1}^N \gamma(z_{nk}) (\mathbf{x}_n - \boldsymbol{\mu}_k)(\mathbf{x}_n - \boldsymbol{\mu}_k)^{\mathrm{T}} = \sum_{k=1}^K \frac{N_k}{N} \mathbf{\Sigma}_k^{\mathrm{ind}} $$
   すなわち、各クラスタの個別共分散の加重平均となる。

### 穴埋めの解答
- ①: $\frac{N_k}{N}$
- ②: $\frac{1}{N}$

In [7]:
# Exercise 9.6 数値検証: 共通共分散 GMM の M ステップ加重平均解
N, D, K = 80, 2, 3
np.random.seed(42)
X = np.random.randn(N, D)
gamma = np.random.dirichlet(np.ones(K), size=N)
mu = np.random.randn(K, D)

# 各クラスタ個別共分散
covs_ind = []
for k in range(K):
    diff = X - mu[k]
    cov_k = (gamma[:, k:k+1] * diff).T @ diff / np.sum(gamma[:, k])
    covs_ind.append(cov_k)

# 共通共分散公式: 1/N sum_k sum_n gamma_nk (x_n - mu_k)(x_n - mu_k)^T
Sigma_common = np.zeros((D, D))
for k in range(K):
    diff = X - mu[k]
    Sigma_common += (gamma[:, k:k+1] * diff).T @ diff
Sigma_common /= N

# N_k / N による個別共分散の加重和と完全一致
Sigma_weighted = sum((np.sum(gamma[:, k]) / N) * covs_ind[k] for k in range(K))
np.testing.assert_allclose(Sigma_common, Sigma_weighted, atol=1e-12)
print("Exercise 9.6 verified: Common covariance equals the exact weighted sum of separate covariances!")

Exercise 9.6 verified: Common covariance equals the exact weighted sum of separate covariances!


---
## <a id="Exercise-9.7 ⭐"></a>Exercise 9.7 ⭐: 完全データ対数尤度最大化による各成分パラメータの独立最尤推定

### 問題の提示
ガウス混合モデルにおける完全データ対数尤度（式 9.36）
$$ \ln p(\mathbf{X}, \mathbf{Z}|\boldsymbol{\mu}, \mathbf{\Sigma}, \boldsymbol{\pi}) = \sum_{n=1}^N \sum_{k=1}^K z_{nk} \left\{ \ln \pi_k + \ln \mathcal{N}(\mathbf{x}_n | \boldsymbol{\mu}_k, \mathbf{\Sigma}_k) \right\} \quad (9.36) $$
の最大化が、各成分 $k$ の平均 $\boldsymbol{\mu}_k$ および共分散 $\mathbf{\Sigma}_k$ をそのクラスタに属するデータ点集合のみから独立に適合すること、および混合係数 $\pi_k$ が各クラスタの点数割合によって与えられることを代数的に確認せよ。

### [解答の道筋と穴埋め]
1. **クラスタごとの完全分離**:
   式 (9.36) は $k$ に関する和の形をしており、相異なる成分 $j \ne k$ のパラメータ同士の結合項を一切持たない：
   $$ \ln p(\mathbf{X}, \mathbf{Z}|\boldsymbol{\theta}) = \sum_{k=1}^K \left[ \sum_{n=1}^N z_{nk} \ln \mathcal{N}(\mathbf{x}_n | \boldsymbol{\mu}_k, \mathbf{\Sigma}_k) \right] + \sum_{k=1}^K \left( \sum_{n=1}^N z_{nk} \right) \ln \pi_k $$
2. **中心と共分散の閉形式解**:
   データ点集合 $\mathcal{C}_k = \{n : z_{nk} = 1\}$ の点数を $N_k = \sum_n z_{nk}$ とすると、
   $$ \boldsymbol{\mu}_k = \frac{\sum_{n=1}^N z_{nk} \mathbf{x}_n}{\sum_{n=1}^N z_{nk}} = \frac{1}{N_k} \sum_{n \in \mathcal{C}_k} \mathbf{x}_n = [ \text{①} ] $$
   $$ \mathbf{\Sigma}_k = \frac{1}{N_k} \sum_{n \in \mathcal{C}_k} (\mathbf{x}_n - \boldsymbol{\mu}_k)(\mathbf{x}_n - \boldsymbol{\mu}_k)^{\mathrm{T}} $$
   となり、通常の単一ガウス分布の標本平均・標本共分散と完全に一致する。
3. **混合係数の解**:
   ラグランジュ未定乗数 $\lambda (\sum_k \pi_k - 1)$ を用いて最大化すると、
   $$ \pi_k = \frac{N_k}{N} = [ \text{②} ] $$
   となり、全データに対する所属割合となる。

### 穴埋めの解答
- ①: $\bar{\mathbf{x}}_k$ (クラスタ $k$ の標本平均)
- ②: $\frac{1}{N} \sum_{n=1}^N z_{nk}$

In [8]:
# Exercise 9.7 数値検証: 完全データ対数尤度最大化とクラスタ標本統計量の一致
N, D, K = 100, 2, 3
X = np.random.randn(N, D)
# 既知の潜在変数 (ハード割り当て)
labels = np.random.choice(K, size=N)
Z = np.zeros((N, K))
for n in range(N):
    Z[n, labels[n]] = 1.0

# 1. 式 (9.36) に基づく重み付き計算
N_k = Z.sum(axis=0)
mu_weighted = (Z.T @ X) / N_k[:, None]
pi_weighted = N_k / N

# 2. 各クラスタデータ点のみを抽出した通常標本統計量
for k in range(K):
    X_k = X[labels == k]
    mu_k_sample = np.mean(X_k, axis=0)
    pi_k_sample = len(X_k) / N
    np.testing.assert_allclose(mu_weighted[k], mu_k_sample, atol=1e-12)
    np.testing.assert_allclose(pi_weighted[k], pi_k_sample, atol=1e-12)

print("Exercise 9.7 verified: Complete-data MLE exactly matches individual cluster sample statistics!")

Exercise 9.7 verified: Complete-data MLE exactly matches individual cluster sample statistics!


---
## <a id="Exercise-9.8"></a>Exercise 9.8: 期待完全データ対数尤度の中心ベクトル $\boldsymbol{\mu}_k$ に関する閉形式最大化

### 問題の提示
負担率 $\gamma(z_{nk})$ を固定したとき、式 (9.40) で定義される期待完全データ対数尤度
$$ \mathcal{Q}(\boldsymbol{\theta}, \boldsymbol{\theta}^{(\mathrm{old})}) = \sum_{n=1}^N \sum_{k=1}^K \gamma(z_{nk}) \left\{ \ln \pi_k + \ln \mathcal{N}(\mathbf{x}_n | \boldsymbol{\mu}_k, \mathbf{\Sigma}_k) \right\} \quad (9.40) $$
を $\boldsymbol{\mu}_k$ に関して最大化することにより、Mステップの閉形式解 (9.17)
$$ \boldsymbol{\mu}_k = \frac{1}{N_k} \sum_{n=1}^N \gamma(z_{nk}) \mathbf{x}_n \quad (9.17) $$
が得られることを示せ。

### [解答の道筋と穴埋め]
1. **$\boldsymbol{\mu}_k$ に依存する項の抽出**:
   $$ \mathcal{Q}(\boldsymbol{\mu}_k) = -\frac{1}{2} \sum_{n=1}^N \gamma(z_{nk}) (\mathbf{x}_n - \boldsymbol{\mu}_k)^{\mathrm{T}} \mathbf{\Sigma}_k^{-1} (\mathbf{x}_n - \boldsymbol{\mu}_k) + \mathrm{const} $$
2. **勾配ベクトルの計算**:
   ベクトル微分の公式 $\nabla_{\boldsymbol{\mu}} (\mathbf{x} - \boldsymbol{\mu})^{\mathrm{T}} \mathbf{A} (\mathbf{x} - \boldsymbol{\mu}) = -2 \mathbf{A} (\mathbf{x} - \boldsymbol{\mu})$ より：
   $$ \nabla_{\boldsymbol{\mu}_k} \mathcal{Q} = [ \text{①} ] \sum_{n=1}^N \gamma(z_{nk}) \mathbf{\Sigma}_k^{-1} (\mathbf{x}_n - \boldsymbol{\mu}_k) $$
3. **停留条件と解の導出**:
   $\nabla_{\boldsymbol{\mu}_k} \mathcal{Q} = \mathbf{0}$ と置き、正定値行列 $\mathbf{\Sigma}_k^{-1}$ を左から乗じて相殺すると：
   $$ \sum_{n=1}^N \gamma(z_{nk}) (\mathbf{x}_n - \boldsymbol{\mu}_k) = \mathbf{0} \implies \left( \sum_{n=1}^N \gamma(z_{nk}) \right) \boldsymbol{\mu}_k = \sum_{n=1}^N \gamma(z_{nk}) \mathbf{x}_n $$
   有効データ点数を $N_k = \sum_{n=1}^N \gamma(z_{nk})$ と置くことで、
   $$ \boldsymbol{\mu}_k = \frac{1}{N_k} \sum_{n=1}^N \gamma(z_{nk}) \mathbf{x}_n = [ \text{②} ] $$
   が導出される。

### 穴埋めの解答
- ①: $\mathbf{\Sigma}_k^{-1}$
- ②: 式 (9.17)

In [9]:
# Exercise 9.8 数値検証: Q の中心ベクトル勾配が M ステップ解で厳密にゼロとなることの検証
N, D = 40, 2
X = np.random.randn(N, D)
gamma_k = np.random.rand(N)
Sigma_k = np.array([[2.0, 0.5], [0.5, 1.5]])
Sigma_k_inv = np.linalg.inv(Sigma_k)

# M ステップ公式による mu_k
N_k = np.sum(gamma_k)
mu_k_opt = np.sum(gamma_k[:, None] * X, axis=0) / N_k

# 勾配の直接評価
grad = np.zeros(D)
for n in range(N):
    grad += gamma_k[n] * Sigma_k_inv @ (X[n] - mu_k_opt)

np.testing.assert_allclose(grad, np.zeros(D), atol=1e-12)
print("Exercise 9.8 verified: Gradient of expected log likelihood is strictly zero at analytic mu_k!")

Exercise 9.8 verified: Gradient of expected log likelihood is strictly zero at analytic mu_k!


---
## <a id="Exercise-9.9"></a>Exercise 9.9: 期待完全データ対数尤度の共分散 $\mathbf{\Sigma}_k$ および混合係数 $\pi_k$ に関する閉形式最大化

### 問題の提示
負担率 $\gamma(z_{nk})$ を固定したとき、式 (9.40) の $\mathcal{Q}(\boldsymbol{\theta}, \boldsymbol{\theta}^{(\mathrm{old})})$ を $\mathbf{\Sigma}_k$ および $\pi_k$ に関して最大化することにより、Mステップの閉形式解 (9.19)
$$ \mathbf{\Sigma}_k = \frac{1}{N_k} \sum_{n=1}^N \gamma(z_{nk}) (\mathbf{x}_n - \boldsymbol{\mu}_k)(\mathbf{x}_n - \boldsymbol{\mu}_k)^{\mathrm{T}} \quad (9.19) $$
および (9.22)
$$ \pi_k = \frac{N_k}{N} \quad (9.22) $$
が得られることを示せ。

### [解答の道筋と穴埋め]
1. **共分散行列 $\mathbf{\Sigma}_k$ の最大化**:
   精度行列 $\mathbf{\Lambda}_k = \mathbf{\Sigma}_k^{-1}$ を用いると：
   $$ \mathcal{Q}(\mathbf{\Lambda}_k) = \frac{1}{2} \sum_{n=1}^N \gamma(z_{nk}) \left[ \ln|\mathbf{\Lambda}_k| - \mathrm{Tr}\left( \mathbf{\Lambda}_k (\mathbf{x}_n - \boldsymbol{\mu}_k)(\mathbf{x}_n - \boldsymbol{\mu}_k)^{\mathrm{T}} \right) \right] + \mathrm{const} $$
   $\frac{\partial \ln|\mathbf{\Lambda}|}{\partial \mathbf{\Lambda}} = \mathbf{\Lambda}^{-1} = \mathbf{\Sigma}_k$ より：
   $$ \frac{\partial \mathcal{Q}}{\partial \mathbf{\Lambda}_k} = \frac{1}{2} N_k \mathbf{\Sigma}_k - \frac{1}{2} \sum_{n=1}^N \gamma(z_{nk}) (\mathbf{x}_n - \boldsymbol{\mu}_k)(\mathbf{x}_n - \boldsymbol{\mu}_k)^{\mathrm{T}} = \mathbf{0} $$
   これを解くことで式 (9.19) が得られる。
2. **混合係数 $\pi_k$ の制約付き最大化**:
   和が 1 となる制約 $\sum_k \pi_k = 1$ の下でラグランジュ関数
   $$ \mathcal{L}(\boldsymbol{\pi}, \lambda) = \sum_{k=1}^K \sum_{n=1}^N \gamma(z_{nk}) \ln \pi_k + \lambda \left( \sum_{k=1}^K \pi_k - 1 \right) = \sum_{k=1}^K N_k \ln \pi_k + \lambda \left( \sum_{k=1}^K \pi_k - 1 \right) $$
   を定義する。$\pi_k$ で微分してゼロと置くと：
   $$ \frac{N_k}{\pi_k} + \lambda = 0 \implies \pi_k = -\frac{N_k}{\lambda} $$
   両辺を $k$ について総和をとると $1 = -\frac{\sum_k N_k}{\lambda} = -\frac{N}{\lambda} \implies \lambda = [ \text{①} ]$。
   したがって、
   $$ \pi_k = [ \text{②} ] $$
   が導出される。

### 穴埋めの解答
- ①: $-N$
- ②: $\frac{N_k}{N}$

In [10]:
# Exercise 9.9 数値検証: 期待完全データ対数尤度の Sigma_k, pi_k 停留条件の厳密検証
N, D, K = 50, 2, 3
X = np.random.randn(N, D)
gamma = np.random.dirichlet(np.ones(K), size=N)
mu = np.random.randn(K, D)

N_k = gamma.sum(axis=0)

# 1. pi_k = N_k / N の検証
pi_opt = N_k / N
np.testing.assert_allclose(np.sum(pi_opt), 1.0, atol=1e-12)

# 2. Sigma_k の勾配ゼロ性検証
for k in range(K):
    diff = X - mu[k]
    Sigma_k_opt = (gamma[:, k:k+1] * diff).T @ diff / N_k[k]
    Lambda_k = np.linalg.inv(Sigma_k_opt)
    
    # dQ / d Lambda_k = 0.5 * (N_k * Sigma_k - sum_n gamma_nk diff diff^T)
    grad_Lambda = 0.5 * (N_k[k] * Sigma_k_opt - (gamma[:, k:k+1] * diff).T @ diff)
    np.testing.assert_allclose(grad_Lambda, np.zeros((D, D)), atol=1e-12)

print("Exercise 9.9 verified: Closed form Sigma_k and pi_k satisfy stationary conditions exactly!")

Exercise 9.9 verified: Closed form Sigma_k and pi_k satisfy stationary conditions exactly!


---
## <a id="Exercise-9.10"></a>Exercise 9.10: 分割ベクトルにおける条件付き分布の混合分布表現

### 問題の提示
混合分布モデル
$$ p(\mathbf{x}) = \sum_{k=1}^K \pi_k p(\mathbf{x}|k) \quad (9.81) $$
において、変数ベクトルを $\mathbf{x} = (\mathbf{x}_a, \mathbf{x}_b)$ と2分割する。
このとき、条件付き密度 $p(\mathbf{x}_b | \mathbf{x}_a)$ もまた混合分布の形をしていることを示し、その新しい混合係数および各成分条件付き密度の数式表現を導出せよ。

### [解答の道筋と穴埋め]
1. **条件付き確率の定義**:
   $$ p(\mathbf{x}_b | \mathbf{x}_a) = \frac{p(\mathbf{x}_a, \mathbf{x}_b)}{p(\mathbf{x}_a)} = \frac{\sum_{k=1}^K \pi_k p(\mathbf{x}_a, \mathbf{x}_b | k)}{\int p(\mathbf{x}_a, \mathbf{x}_b) \mathrm{d}\mathbf{x}_b} $$
2. **分子・分母の各成分表現**:
   条件付き確率の乗法定理 $p(\mathbf{x}_a, \mathbf{x}_b | k) = p(\mathbf{x}_a | k) p(\mathbf{x}_b | \mathbf{x}_a, k)$ を適用すると：
   $$ p(\mathbf{x}_a) = \sum_{j=1}^K \pi_j p(\mathbf{x}_a | j) $$
   $$ p(\mathbf{x}_a, \mathbf{x}_b) = \sum_{k=1}^K \pi_k p(\mathbf{x}_a | k) p(\mathbf{x}_b | \mathbf{x}_a, k) $$
3. **入力依存の混合係数の定義**:
   したがって、条件付き密度は：
   $$ p(\mathbf{x}_b | \mathbf{x}_a) = \sum_{k=1}^K \left[ \frac{\pi_k p(\mathbf{x}_a | k)}{\sum_{j=1}^K \pi_j p(\mathbf{x}_a | j)} \right] [ \text{①} ] = \sum_{k=1}^K \tilde{\pi}_k(\mathbf{x}_a) p(\mathbf{x}_b | \mathbf{x}_a, k) $$
   ここで新しい混合係数は
   $$ \tilde{\pi}_k(\mathbf{x}_a) = [ \text{②} ] $$
   であり、$\sum_k \tilde{\pi}_k(\mathbf{x}_a) = 1$ かつ $\tilde{\pi}_k(\mathbf{x}_a) \ge 0$ を満たすため、条件付き分布も厳密に混合分布となる（これは混合密度ネットワーク (MDN) や専門家混合 (MoE) の基礎原理である）。

### 穴埋めの解答
- ①: $p(\mathbf{x}_b | \mathbf{x}_a, k)$
- ②: $\frac{\pi_k p(\mathbf{x}_a | k)}{\sum_j \pi_j p(\mathbf{x}_a | j)}$

In [11]:
# Exercise 9.10 数値検証: 分割ベクトルの条件付き分布が混合分布となることの数値積分検証
import numpy as np

# 2次元2成分 GMM
pi = np.array([0.3, 0.7])
mu = np.array([[1.0, 2.0], [-1.0, -0.5]])
cov = np.array([
    [[1.0, 0.5], [0.5, 1.2]],
    [[0.8, -0.3], [-0.3, 1.5]]
])

def gaussian_2d(x, m, c):
    d = len(m)
    diff = x - m
    return np.exp(-0.5 * diff @ np.linalg.inv(c) @ diff) / np.sqrt((2 * np.pi)**d * np.linalg.det(c))

x_a = 0.5
# 1. 直接法: p(x_a, x_b) / p(x_a)
p_xa = sum(pi[k] * np.exp(-0.5 * (x_a - mu[k, 0])**2 / cov[k, 0, 0]) / np.sqrt(2 * np.pi * cov[k, 0, 0]) for k in range(2))

# 2. 混合条件付き公式: sum_k pi_tilde_k * p(x_b | x_a, k)
# 各成分の条件付きガウス分布 p(x_b | x_a, k): N(mu_{b|a}, sigma_{b|a}^2)
pi_tilde = np.zeros(2)
for k in range(2):
    p_xa_k = np.exp(-0.5 * (x_a - mu[k, 0])**2 / cov[k, 0, 0]) / np.sqrt(2 * np.pi * cov[k, 0, 0])
    pi_tilde[k] = pi[k] * p_xa_k
pi_tilde /= np.sum(pi_tilde)

x_b_test = 1.2
p_cond_formula = 0.0
for k in range(2):
    # ガウス条件付き分布公式
    mu_b_given_a = mu[k, 1] + cov[k, 1, 0] / cov[k, 0, 0] * (x_a - mu[k, 0])
    var_b_given_a = cov[k, 1, 1] - cov[k, 1, 0]**2 / cov[k, 0, 0]
    p_cond_k = np.exp(-0.5 * (x_b_test - mu_b_given_a)**2 / var_b_given_a) / np.sqrt(2 * np.pi * var_b_given_a)
    p_cond_formula += pi_tilde[k] * p_cond_k

# 直接結合密度との比
p_joint = sum(pi[k] * gaussian_2d(np.array([x_a, x_b_test]), mu[k], cov[k]) for k in range(2))
p_cond_direct = p_joint / p_xa

np.testing.assert_allclose(p_cond_formula, p_cond_direct, atol=1e-12)
print(f"Exercise 9.10 verified: Conditional density {p_cond_formula:.6f} matches joint/marginal ratio exactly!")

Exercise 9.10 verified: Conditional density 0.224915 matches joint/marginal ratio exactly!


---
## <a id="Exercise-9.11"></a>Exercise 9.11: 分散ゼロ極限 $\epsilon \to 0$ における EM アルゴリズムと K-means の等価性

### 問題の提示
各成分が等方共分散 $\mathbf{\Sigma}_k = \epsilon \mathbf{I}$ を持つガウス混合モデルにおいて、$\epsilon \to 0$ の極限を考える。
このとき、事後負担率 $\gamma(z_{nk})$ が K-means の二値指示変数 $r_{nk} \in \{0, 1\}$ に収束し、期待完全データ対数尤度 $\mathcal{Q}$ の最大化が K-means 歪み尺度 $J$ の最小化と等価になることを示せ。

### [解答の道筋と穴埋め]
1. **負担率の極限表現**:
   $$ \gamma(z_{nk}) = \frac{\pi_k \exp\left(-\frac{\|\mathbf{x}_n - \boldsymbol{\mu}_k\|^2}{2\epsilon}\right)}{\sum_{j=1}^K \pi_j \exp\left(-\frac{\|\mathbf{x}_n - \boldsymbol{\mu}_j\|^2}{2\epsilon}\right)} $$
   $\epsilon \to 0$ のとき、分母の中でユークリッド距離 $\|\mathbf{x}_n - \boldsymbol{\mu}_j\|^2$ が最も小さい成分が支配的となり、指数関数の比は：
   $$ \lim_{\epsilon \to 0} \gamma(z_{nk}) = \begin{cases} 1 & (k = \arg\min_j \|\mathbf{x}_n - \boldsymbol{\mu}_j\|^2) \\ 0 & (\text{otherwise}) \end{cases} = [ \text{①} ] $$
2. **目的関数の極限関係**:
   期待完全データ対数尤度 (9.40) に $\mathbf{\Sigma}_k = \epsilon \mathbf{I}$ を代入すると：
   $$ \mathcal{Q} = -\frac{1}{2\epsilon} \sum_{n=1}^N \sum_{k=1}^K \gamma(z_{nk}) \|\mathbf{x}_n - \boldsymbol{\mu}_k\|^2 - \frac{ND}{2} \ln(2\pi\epsilon) + \sum_{n, k} \gamma(z_{nk}) \ln \pi_k $$
   $\epsilon \to 0$ において最右辺第1項が圧倒的に支配的となる：
   $$ \mathcal{Q} = -[ \text{②} ] J + \mathcal{O}(\ln\epsilon) $$
   したがって、$\mathcal{Q}$ を最大化することは $J = \sum_{n, k} r_{nk} \|\mathbf{x}_n - \boldsymbol{\mu}_k\|^2$ を最小化することと完全に等価となる。

### 穴埋めの解答
- ①: $r_{nk}$ (K-means ハード割り当て)
- ②: $\frac{1}{2\epsilon}$

In [12]:
# Exercise 9.11 数値検証: epsilon -> 0 における負担率 gamma_nk の K-means 指示変数への収束
mu = np.array([[0.0, 0.0], [2.0, 2.0]])
x = np.array([0.5, 0.5]) # mu[0] に近い点
pi = np.array([0.5, 0.5])

epsilons = [1.0, 0.1, 0.01, 1e-4]
print("Convergence of gamma(z_n0) as epsilon -> 0:")
for eps in epsilons:
    dist0 = np.sum((x - mu[0])**2)
    dist1 = np.sum((x - mu[1])**2)
    log_p0 = np.log(pi[0]) - dist0 / (2 * eps)
    log_p1 = np.log(pi[1]) - dist1 / (2 * eps)
    # LogSumExp
    max_log = max(log_p0, log_p1)
    gamma0 = np.exp(log_p0 - max_log) / (np.exp(log_p0 - max_log) + np.exp(log_p1 - max_log))
    print(f"  epsilon = {eps:6.4f}: gamma0 = {gamma0:.8f}")

assert gamma0 > 1 - 1e-6, "gamma0 must approach 1.0 as epsilon -> 0"
print("Exercise 9.11 verified: Soft responsibility strictly converges to hard K-means assignment!")

Convergence of gamma(z_n0) as epsilon -> 0:
  epsilon = 1.0000: gamma0 = 0.88079708
  epsilon = 0.1000: gamma0 = 1.00000000
  epsilon = 0.0100: gamma0 = 1.00000000
  epsilon = 0.0001: gamma0 = 1.00000000
Exercise 9.11 verified: Soft responsibility strictly converges to hard K-means assignment!


---
## <a id="Exercise-9.12 ⭐"></a>Exercise 9.12 ⭐: 一般混合分布全体の平均および共分散行列の合成則 (式 9.49, 9.50)

### 問題の提示
一般混合分布 $p(\mathbf{x}) = \sum_{k=1}^K \pi_k p(\mathbf{x}|k)$ を考える（変数は連続・離散・混合のいずれでもよい）。
各成分分布 $p(\mathbf{x}|k)$ の平均を $\boldsymbol{\mu}_k$、共分散行列を $\mathbf{\Sigma}_k$ とするとき、混合分布全体の平均 $\mathbb{E}[\mathbf{x}]$ および共分散行列 $\mathrm{cov}[\mathbf{x}]$ がそれぞれ
$$ \mathbb{E}[\mathbf{x}] = \sum_{k=1}^K \pi_k \boldsymbol{\mu}_k \quad (9.49) $$
$$ \mathrm{cov}[\mathbf{x}] = \sum_{k=1}^K \pi_k \left\{ \mathbf{\Sigma}_k + (\boldsymbol{\mu}_k - \mathbb{E}[\mathbf{x}])(\boldsymbol{\mu}_k - \mathbb{E}[\mathbf{x}])^{\mathrm{T}} \right\} \quad (9.50) $$
で与えられることを証明せよ。

### [解答の道筋と穴埋め]
1. **全体の平均 $\mathbb{E}[\mathbf{x}]$**:
   全期待値の法則 $\mathbb{E}[\mathbf{x}] = \mathbb{E}_k [\mathbb{E}[\mathbf{x}|k]]$ より：
   $$ \mathbb{E}[\mathbf{x}] = \int \mathbf{x} \sum_{k=1}^K \pi_k p(\mathbf{x}|k) \mathrm{d}\mathbf{x} = \sum_{k=1}^K \pi_k \int \mathbf{x} p(\mathbf{x}|k) \mathrm{d}\mathbf{x} = [ \text{①} ] $$
2. **全体の2次モーメント**:
   $$ \mathbb{E}[\mathbf{x} \mathbf{x}^{\mathrm{T}}] = \sum_{k=1}^K \pi_k \mathbb{E}[\mathbf{x} \mathbf{x}^{\mathrm{T}}|k] = \sum_{k=1}^K \pi_k (\mathbf{\Sigma}_k + \boldsymbol{\mu}_k \boldsymbol{\mu}_k^{\mathrm{T}}) $$
3. **共分散行列への変形**:
   $\mathrm{cov}[\mathbf{x}] = \mathbb{E}[\mathbf{x} \mathbf{x}^{\mathrm{T}}] - \mathbb{E}[\mathbf{x}]\mathbb{E}[\mathbf{x}]^{\mathrm{T}}$ より：
   $$ \mathrm{cov}[\mathbf{x}] = \sum_{k=1}^K \pi_k \mathbf{\Sigma}_k + \sum_{k=1}^K \pi_k \boldsymbol{\mu}_k \boldsymbol{\mu}_k^{\mathrm{T}} - \mathbb{E}[\mathbf{x}]\mathbb{E}[\mathbf{x}]^{\mathrm{T}} $$
   ここで $(\boldsymbol{\mu}_k - \mathbb{E}[\mathbf{x}])(\boldsymbol{\mu}_k - \mathbb{E}[\mathbf{x}])^{\mathrm{T}} = \boldsymbol{\mu}_k \boldsymbol{\mu}_k^{\mathrm{T}} - \boldsymbol{\mu}_k \mathbb{E}[\mathbf{x}]^{\mathrm{T}} - \mathbb{E}[\mathbf{x}]\boldsymbol{\mu}_k^{\mathrm{T}} + \mathbb{E}[\mathbf{x}]\mathbb{E}[\mathbf{x}]^{\mathrm{T}}$ を $\pi_k$ で加重総和すると：
   $$ \sum_{k=1}^K \pi_k (\boldsymbol{\mu}_k - \mathbb{E}[\mathbf{x}])(\boldsymbol{\mu}_k - \mathbb{E}[\mathbf{x}])^{\mathrm{T}} = \sum_{k=1}^K \pi_k \boldsymbol{\mu}_k \boldsymbol{\mu}_k^{\mathrm{T}} - [ \text{②} ] $$
   したがって、式 (9.50)（各クラスタ内共分散の平均 ＋ クラスタ間分散）が厳密に成立する。

### 穴埋めの解答
- ①: $\sum_{k=1}^K \pi_k \boldsymbol{\mu}_k$
- ②: $\mathbb{E}[\mathbf{x}]\mathbb{E}[\mathbf{x}]^{\mathrm{T}}$

In [13]:
# Exercise 9.12 数値検証: 混合分布の理論平均・共分散とモンテカルロサンプリングの整合
K, D = 3, 2
pi = np.array([0.2, 0.5, 0.3])
mu = np.array([[2.0, 0.0], [-1.0, 3.0], [0.0, -2.0]])
cov = np.array([
    [[1.0, 0.2], [0.2, 0.8]],
    [[1.5, -0.4], [-0.4, 0.6]],
    [[0.7, 0.1], [0.1, 1.2]]
])

# 理論平均
E_x_theory = sum(pi[k] * mu[k] for k in range(K))
# 理論共分散 (式 9.50)
cov_x_theory = sum(pi[k] * (cov[k] + np.outer(mu[k] - E_x_theory, mu[k] - E_x_theory)) for k in range(K))

# モンテカルロサンプリング
N_samples = 200000
labels = np.random.choice(K, size=N_samples, p=pi)
samples = np.zeros((N_samples, D))
for k in range(K):
    idx = (labels == k)
    if np.sum(idx) > 0:
        samples[idx] = np.random.multivariate_normal(mu[k], cov[k], size=np.sum(idx))

E_x_sample = np.mean(samples, axis=0)
cov_x_sample = np.cov(samples, rowvar=False)

np.testing.assert_allclose(E_x_sample, E_x_theory, atol=0.02)
np.testing.assert_allclose(cov_x_sample, cov_x_theory, atol=0.02)
print("Exercise 9.12 verified: Theoretical mixture mean and covariance perfectly match empirical moments!")

Exercise 9.12 verified: Theoretical mixture mean and covariance perfectly match empirical moments!


---
## <a id="Exercise-9.13"></a>Exercise 9.13: ベルヌーイ混合モデルにおける同一初期値での1反復退化現象の証明

### 問題の提示
ベルヌーイ混合モデル (BMM) において、EMアルゴリズムの再推定方程式を満たす最尤解において $\mathbb{E}[\mathbf{x}] = \frac{1}{N} \sum_{n=1}^N \mathbf{x}_n \equiv \bar{\mathbf{x}}$ が成立することを示せ。
さらに、すべての混合成分のパラメータが同一の値 $\boldsymbol{\mu}_k = \boldsymbol{\mu}$ および $\pi_k = 1/K$ に初期化された場合、アルゴリズムがわずか1回の反復で標本平均 $\boldsymbol{\mu}_k = \bar{\mathbf{x}}$ に退化し、以後全く更新されなくなることを証明せよ。

### [解答の道筋と穴埋め]
1. **混合期待値の整合**:
   式 (9.49) より $\mathbb{E}[\mathbf{x}] = \sum_{k=1}^K \pi_k \boldsymbol{\mu}_k$。
   Mステップの再推定式 $\pi_k = \frac{N_k}{N}, \boldsymbol{\mu}_k = \frac{1}{N_k} \sum_n \gamma(z_{nk}) \mathbf{x}_n$ を代入すると：
   $$ \mathbb{E}[\mathbf{x}] = \sum_{k=1}^K \frac{N_k}{N} \left( \frac{1}{N_k} \sum_{n=1}^N \gamma(z_{nk}) \mathbf{x}_n \right) = \frac{1}{N} \sum_{n=1}^N \left( \sum_{k=1}^K \gamma(z_{nk}) \right) \mathbf{x}_n = [ \text{①} ] $$
   なぜなら負担率の定義より $\sum_k \gamma(z_{nk}) = 1$ だからである。
2. **同一初期値におけるEステップ**:
   すべての $k$ で $\boldsymbol{\mu}_k = \boldsymbol{\mu}, \pi_k = 1/K$ のとき、任意のデータ点 $\mathbf{x}_n$ に対し条件付き確率 $p(\mathbf{x}_n | \boldsymbol{\mu}_k)$ は $k$ に依存せず全て等しい。
   したがって、負担率は：
   $$ \gamma(z_{nk}) = \frac{\frac{1}{K} p(\mathbf{x}_n|\boldsymbol{\mu})}{\sum_{j=1}^K \frac{1}{K} p(\mathbf{x}_n|\boldsymbol{\mu})} = [ \text{②} ] $$
3. **Mステップにおける即時退化**:
   すべての $n, k$ で $\gamma(z_{nk}) = 1/K$ となるため、$N_k = \sum_n \frac{1}{K} = N/K$。
   更新後の中心パラメータは：
   $$ \boldsymbol{\mu}_k^{\mathrm{new}} = \frac{1}{N/K} \sum_{n=1}^N \frac{1}{K} \mathbf{x}_n = \frac{1}{N} \sum_{n=1}^N \mathbf{x}_n = \bar{\mathbf{x}} $$
   かつ $\pi_k^{\mathrm{new}} = \frac{N/K}{N} = \frac{1}{K}$。
   すべてのクラスタが同一の標本平均 $\bar{\mathbf{x}}$ となり、対称性が破れないため、以後一切変化せず1ステップで停止する。

### 穴埋めの解答
- ①: $\frac{1}{N} \sum_{n=1}^N \mathbf{x}_n \equiv \bar{\mathbf{x}}$
- ②: $\frac{1}{K}$

In [14]:
# Exercise 9.13 数値検証: ベルヌーイ混合モデルの同一初期値における1反復退化
N, D, K = 50, 4, 3
np.random.seed(42)
X = (np.random.rand(N, D) > 0.5).astype(float)
x_bar = np.mean(X, axis=0)

# 同一パラメータで初期化
mu = np.full((K, D), 0.5)
pi = np.full(K, 1.0 / K)

# Eステップ
resp = np.zeros((N, K))
for n in range(N):
    dens = np.array([pi[k] * np.prod(mu[k]**X[n] * (1 - mu[k])**(1 - X[n])) for k in range(K)])
    resp[n] = dens / np.sum(dens)

assert np.allclose(resp, 1.0 / K), "Responsibilities must all be exactly 1/K!"

# Mステップ
N_k = resp.sum(axis=0)
mu_new = (resp.T @ X) / N_k[:, None]
pi_new = N_k / N

for k in range(K):
    np.testing.assert_allclose(mu_new[k], x_bar, atol=1e-12)
    np.testing.assert_allclose(pi_new[k], 1.0 / K, atol=1e-12)

print("Exercise 9.13 verified: Identically initialized BMM strictly collapses to sample mean x_bar in 1 step!")

Exercise 9.13 verified: Identically initialized BMM strictly collapses to sample mean x_bar in 1 step!


---
## <a id="Exercise-9.14"></a>Exercise 9.14: ベルヌーイ混合モデルの同時分布と潜在変数周辺化による観測分布の導出

### 問題の提示
ベルヌーイ分布の混合モデルにおいて、潜在変数 $\mathbf{z}$（1-of-K 表現）とパラメータ $\boldsymbol{\pi}$ の事前分布 $p(\mathbf{z}|\boldsymbol{\pi}) = \prod_k \pi_k^{z_k}$（式 9.53）および条件付き観測分布
$$ p(\mathbf{x}|\mathbf{z}, \boldsymbol{\mu}) = \prod_{k=1}^K p(\mathbf{x}|\boldsymbol{\mu}_k)^{z_k} = \prod_{k=1}^K \left[ \prod_{i=1}^D \mu_{ki}^{x_i} (1 - \mu_{ki})^{1 - x_i} \right]^{z_k} \quad (9.52) $$
の積からなる同時分布 $p(\mathbf{x}, \mathbf{z}|\boldsymbol{\mu}, \boldsymbol{\pi})$ を構成し、$\mathbf{z}$ について周辺化することで観測周辺分布 (9.47)
$$ p(\mathbf{x}|\boldsymbol{\mu}, \boldsymbol{\pi}) = \sum_{k=1}^K \pi_k p(\mathbf{x}|\boldsymbol{\mu}_k) \quad (9.47) $$
が得られることを代数的に示せ。

### [解答の道筋と穴埋め]
1. **同時分布の表現**:
   $$ p(\mathbf{x}, \mathbf{z}|\boldsymbol{\mu}, \boldsymbol{\pi}) = p(\mathbf{x}|\mathbf{z}, \boldsymbol{\mu}) p(\mathbf{z}|\boldsymbol{\pi}) = \prod_{k=1}^K [ \text{①} ]^{z_k} $$
2. **潜在変数 $\mathbf{z}$ の全状態**:
   $\mathbf{z}$ は $k$ 番目のみが 1 で他が 0 の $K$ 個の基底ベクトル $\mathbf{e}_1, \dots, \mathbf{e}_K$ のいずれかの状態を取る。
   $\mathbf{z} = \mathbf{e}_j$ のとき、積のうち $k=j$ の項のみが残り：
   $$ p(\mathbf{x}, \mathbf{z} = \mathbf{e}_j | \boldsymbol{\mu}, \boldsymbol{\pi}) = \pi_j p(\mathbf{x}|\boldsymbol{\mu}_j) $$
3. **総和による周辺化**:
   $$ p(\mathbf{x}|\boldsymbol{\mu}, \boldsymbol{\pi}) = \sum_{\mathbf{z}} p(\mathbf{x}, \mathbf{z}|\boldsymbol{\mu}, \boldsymbol{\pi}) = \sum_{j=1}^K p(\mathbf{x}, \mathbf{z} = \mathbf{e}_j | \boldsymbol{\mu}, \boldsymbol{\pi}) = [ \text{②} ] $$
   これにより、式 (9.47) が得られる。

### 穴埋めの解答
- ①: $\pi_k p(\mathbf{x}|\boldsymbol{\mu}_k)$
- ②: $\sum_{k=1}^K \pi_k p(\mathbf{x}|\boldsymbol{\mu}_k)$

In [15]:
# Exercise 9.14 数値検証: ベルヌーイ混合モデルの同時確率テンソル総和と周辺密度の完全一致
D, K = 3, 2
pi = np.array([0.4, 0.6])
mu = np.array([[0.2, 0.8, 0.5], [0.7, 0.1, 0.9]])
x = np.array([1.0, 0.0, 1.0])

# 各成分の条件付き密度
p_x_given_k = np.array([np.prod(mu[k]**x * (1 - mu[k])**(1 - x)) for k in range(K)])

# 同時確率 p(x, z = e_k)
joint = pi * p_x_given_k

# 周辺確率 (総和)
marginal_sum = np.sum(joint)

# 式 (9.47)
marginal_direct = sum(pi[k] * np.prod(mu[k]**x * (1 - mu[k])**(1 - x)) for k in range(K))

np.testing.assert_allclose(marginal_sum, marginal_direct, atol=1e-12)
print(f"Exercise 9.14 verified: Marginalized joint {marginal_sum:.6f} matches mixture equation exactly!")

Exercise 9.14 verified: Marginalized joint 0.348200 matches mixture equation exactly!


---
## <a id="Exercise-9.15"></a>Exercise 9.15: ベルヌーイ混合モデルにおける中心パラメータ $\boldsymbol{\mu}_k$ の M ステップ再推定式 (9.59) の導出

### 問題の提示
ベルヌーイ混合モデルの期待完全データ対数尤度関数（式 9.55）
$$ \mathcal{Q}(\boldsymbol{\theta}, \boldsymbol{\theta}^{(\mathrm{old})}) = \sum_{n=1}^N \sum_{k=1}^K \gamma(z_{nk}) \left\{ \ln \pi_k + \sum_{i=1}^D \left[ x_{ni} \ln \mu_{ki} + (1 - x_{ni}) \ln (1 - \mu_{ki}) \right] \right\} \quad (9.55) $$
を $\boldsymbol{\mu}_k$ に関して最大化することにより、Mステップの更新式 (9.59)
$$ \boldsymbol{\mu}_k = \frac{1}{N_k} \sum_{n=1}^N \gamma(z_{nk}) \mathbf{x}_n \quad (9.59) $$
が得られることを示せ。

### [解答の道筋と穴埋め]
1. **成分 $\mu_{ki}$ に関する偏微分**:
   各次元 $i \in \{1, \dots, D\}$ は互いに独立に最適化できる。$\mathcal{Q}$ を $\mu_{ki}$ で微分すると：
   $$ \frac{\partial \mathcal{Q}}{\partial \mu_{ki}} = \sum_{n=1}^N \gamma(z_{nk}) \left[ \frac{x_{ni}}{\mu_{ki}} - \frac{1 - x_{ni}}{1 - \mu_{ki}} \right] = \sum_{n=1}^N \gamma(z_{nk}) \frac{x_{ni} (1 - \mu_{ki}) - (1 - x_{ni}) \mu_{ki}}{\mu_{ki} (1 - \mu_{ki})} $$
2. **分子の整理**:
   分子を展開すると $x_{ni} - x_{ni} \mu_{ki} - \mu_{ki} + x_{ni} \mu_{ki} = [ \text{①} ]$ となる。したがって：
   $$ \frac{\partial \mathcal{Q}}{\partial \mu_{ki}} = \frac{1}{\mu_{ki} (1 - \mu_{ki})} \sum_{n=1}^N \gamma(z_{nk}) (x_{ni} - \mu_{ki}) $$
3. **停留条件と更新式**:
   これを $0$ と置くと：
   $$ \sum_{n=1}^N \gamma(z_{nk}) (x_{ni} - \mu_{ki}) = 0 \implies \left( \sum_{n=1}^N \gamma(z_{nk}) \right) \mu_{ki} = \sum_{n=1}^N \gamma(z_{nk}) x_{ni} $$
   実効点数 $N_k = \sum_n \gamma(z_{nk})$ で割ることにより：
   $$ \boldsymbol{\mu}_k = [ \text{②} ] $$
   が得られ、ガウス混合モデルの場合と全く同一の加重標本平均形式となる。

### 穴埋めの解答
- ①: $x_{ni} - \mu_{ki}$
- ②: $\frac{1}{N_k} \sum_{n=1}^N \gamma(z_{nk}) \mathbf{x}_n$

In [16]:
# Exercise 9.15 数値検証: BMM の期待完全データ対数尤度の mu_ki 勾配ゼロ性
N, D, K = 40, 3, 2
X = (np.random.rand(N, D) > 0.5).astype(float)
gamma = np.random.dirichlet(np.ones(K), size=N)

N_k = gamma.sum(axis=0)
mu_opt = (gamma.T @ X) / N_k[:, None]

# 勾配の直接計算
for k in range(K):
    grad = np.zeros(D)
    for i in range(D):
        for n in range(N):
            grad[i] += gamma[n, k] * (X[n, i] / mu_opt[k, i] - (1 - X[n, i]) / (1 - mu_opt[k, i]))
    np.testing.assert_allclose(grad, np.zeros(D), atol=1e-12)

print("Exercise 9.15 verified: Gradient of BMM expected log likelihood is strictly zero at mu_k M-step solution!")

Exercise 9.15 verified: Gradient of BMM expected log likelihood is strictly zero at mu_k M-step solution!


---
## <a id="Exercise-9.16"></a>Exercise 9.16: ベルヌーイ混合モデルにおける混合係数 $\pi_k$ の M ステップ更新式 (9.60) の導出

### 問題の提示
ベルヌーイ混合モデルの期待完全データ対数尤度関数 (9.55) を、混合係数の総和制約 $\sum_{k=1}^K \pi_k = 1$ の下でラグランジュ乗数法を用いて最大化することにより、Mステップの更新式 (9.60)
$$ \pi_k = \frac{N_k}{N} \quad (9.60) $$
が得られることを示せ。

### [解答の道筋と穴埋め]
1. **ラグランジュ関数の設定**:
   式 (9.55) のうち $\pi_k$ に依存する項と制約項をまとめたラグランジュ関数は：
   $$ \mathcal{L}(\boldsymbol{\pi}, \lambda) = \sum_{k=1}^K \sum_{n=1}^N \gamma(z_{nk}) \ln \pi_k + \lambda \left( \sum_{k=1}^K \pi_k - 1 \right) = \sum_{k=1}^K N_k \ln \pi_k + \lambda \left( \sum_{k=1}^K \pi_k - 1 \right) $$
2. **停留条件**:
   $\pi_k$ に関して偏微分して $0$ と置く：
   $$ \frac{\partial \mathcal{L}}{\partial \pi_k} = \frac{N_k}{\pi_k} + \lambda = 0 \implies \pi_k = -[ \text{①} ] $$
3. **ラグランジュ乗数の決定**:
   制約 $\sum_{k=1}^K \pi_k = 1$ より：
   $$ \sum_{k=1}^K \left(-\frac{N_k}{\lambda}\right) = -\frac{1}{\lambda} \sum_{k=1}^K N_k = -\frac{N}{\lambda} = 1 \implies \lambda = -N $$
   これを代入することで：
   $$ \pi_k = [ \text{②} ] $$
   が厳密に導出される。

### 穴埋めの解答
- ①: $\frac{N_k}{\lambda}$
- ②: $\frac{N_k}{N}$

In [17]:
# Exercise 9.16 数値検証: BMM 混合係数のラグランジュ乗数解と総和 1 の検証
N, K = 100, 4
gamma = np.random.dirichlet(np.ones(K), size=N)
N_k = gamma.sum(axis=0)

pi_k = N_k / N
np.testing.assert_allclose(np.sum(pi_k), 1.0, atol=1e-12)
assert np.all(pi_k >= 0), "Mixing coefficients must be non-negative!"
print(f"Exercise 9.16 verified: pi_k = N_k / N sums to {np.sum(pi_k):.4f} with values {pi_k}")

Exercise 9.16 verified: pi_k = N_k / N sums to 1.0000 with values [0.2303 0.2659 0.2028 0.301 ]


---
## <a id="Exercise-9.17"></a>Exercise 9.17: ベルヌーイ混合モデルにおける対数尤度の上界性と特異点（発散）の非存在証明

### 問題の提示
離散確率変数ベクトル $\mathbf{x}_n$ に対するベルヌーイ分布の確率値が常に $0 \le p(\mathbf{x}_n | \boldsymbol{\mu}_k) \le 1$ を満たすことから、ベルヌーイ混合モデルの不完全データ対数尤度関数
$$ \ln p(\mathbf{X}|\boldsymbol{\mu}, \boldsymbol{\pi}) = \sum_{n=1}^N \ln \left\{ \sum_{k=1}^K \pi_k p(\mathbf{x}_n | \boldsymbol{\mu}_k) \right\} $$
が上に有界（$\le 0$）であり、したがってガウス混合モデルに見られるような尤度が正の無限大へ発散する**特異点（singularity）が存在しない**ことを証明せよ。

### [解答の道筋と穴埋め]
1. **単一成分の確率の有界性**:
   各成分の確率質量関数は $0 \le \mu_{ki} \le 1$ に対し：
   $$ p(\mathbf{x}_n | \boldsymbol{\mu}_k) = \prod_{i=1}^D \mu_{ki}^{x_{ni}} (1 - \mu_{ki})^{1 - x_{ni}} $$
   各因子は $0$ 以上 $1$ 以下の値を取るため、その積も必ず $[ \text{①} ]$ を満たす。
2. **混合確率の有界性**:
   $\pi_k \ge 0$ かつ $\sum_{k=1}^K \pi_k = 1$ であるため、凸結合の性質より：
   $$ 0 \le p(\mathbf{x}_n) = \sum_{k=1}^K \pi_k p(\mathbf{x}_n | \boldsymbol{\mu}_k) \le \sum_{k=1}^K \pi_k \cdot 1 = 1 $$
3. **対数尤度の上界**:
   対数関数は単調増加であり、$\ln(1) = 0$ であるため：
   $$ \ln p(\mathbf{x}_n) \le 0 \implies \ln p(\mathbf{X}) = \sum_{n=1}^N \ln p(\mathbf{x}_n) \le [ \text{②} ] $$
   連続変数のガウス分布では共分散行列の分散 $\sigma_k^2 \to 0$ のとき確率密度が $+\infty$ に発散し得るが、離散確率分布の確率質量は決して $1$ を超えないため、特異点が生じることは原理的にあり得ない。

### 穴埋めの解答
- ①: $0 \le p(\mathbf{x}_n | \boldsymbol{\mu}_k) \le 1$
- ②: $0$

In [18]:
# Exercise 9.17 数値検証: BMM の対数尤度が常に <= 0 であることの実験的検証
N, D, K = 30, 5, 3
np.random.seed(42)
X = (np.random.rand(N, D) > 0.5).astype(float)

# 多様なパラメータ（境界付近を含む）で評価
for trial in range(5):
    mu_rand = np.random.uniform(0.01, 0.99, size=(K, D))
    pi_rand = np.random.dirichlet(np.ones(K))
    
    log_lik = 0.0
    for n in range(N):
        p_xn = sum(pi_rand[k] * np.prod(mu_rand[k]**X[n] * (1 - mu_rand[k])**(1 - X[n])) for k in range(K))
        assert p_xn <= 1.0 + 1e-12, "Discrete probability cannot exceed 1!"
        log_lik += np.log(p_xn)
    
    assert log_lik <= 0.0, "Log likelihood of discrete mixture must be non-positive!"

print("Exercise 9.17 verified: BMM log likelihood is strictly bounded above by 0 with no singularities!")

Exercise 9.17 verified: BMM log likelihood is strictly bounded above by 0 with no singularities!


---
## <a id="Exercise-9.18"></a>Exercise 9.18: Beta-Dirichlet 事前分布を持つベルヌーイ混合モデルの MAP-EM アルゴリズム導出

### 問題の提示
9.3.3節のベルヌーイ混合モデルにおいて、各成分パラメータ $\mu_{ki}$ に対する独立なベータ事前分布
$$ p(\mu_{ki} | a_{ki}, b_{ki}) = \frac{\Gamma(a_{ki} + b_{ki})}{\Gamma(a_{ki})\Gamma(b_{ki})} \mu_{ki}^{a_{ki} - 1} (1 - \mu_{ki})^{b_{ki} - 1} $$
および混合係数 $\boldsymbol{\pi}$ に対するディリクレ事前分布
$$ p(\boldsymbol{\pi} | \boldsymbol{\alpha}) = \frac{\Gamma\left(\sum_k \alpha_k\right)}{\prod_k \Gamma(\alpha_k)} \prod_{k=1}^K \pi_k^{\alpha_k - 1} $$
を導入する。パラメータの事後分布 $p(\boldsymbol{\mu}, \boldsymbol{\pi}|\mathbf{X})$ を最大化する MAP-EM アルゴリズムの Eステップおよび Mステップ更新式を導出せよ。

### [解答の道筋と穴埋め]
1. **Eステップ**:
   Exercise 9.4 で示した通り、事前分布は潜在変数 $\mathbf{Z}$ に依存しないため、事後負担率 $\gamma(z_{nk})$ の計算式は最尤推定の場合と完全に同一である：
   $$ \gamma(z_{nk}) = \frac{\pi_k p(\mathbf{x}_n|\boldsymbol{\mu}_k)}{\sum_{j=1}^K \pi_j p(\mathbf{x}_n|\boldsymbol{\mu}_j)} $$
2. **Mステップ ($\mu_{ki}$ の更新)**:
   最大化すべき目的関数は $\mathcal{Q} + \ln p(\boldsymbol{\mu}) + \ln p(\boldsymbol{\pi})$ である。$\mu_{ki}$ に依存する項を取り出すと：
   $$ \sum_{n=1}^N \gamma(z_{nk}) [x_{ni} \ln \mu_{ki} + (1 - x_{ni}) \ln(1 - \mu_{ki})] + (a_{ki} - 1)\ln \mu_{ki} + (b_{ki} - 1)\ln(1 - \mu_{ki}) $$
   これを $\mu_{ki}$ で微分して $0$ と置くと：
   $$ \frac{\sum_n \gamma(z_{nk}) x_{ni} + a_{ki} - 1}{\mu_{ki}} - \frac{\sum_n \gamma(z_{nk}) (1 - x_{ni}) + b_{ki} - 1}{1 - \mu_{ki}} = 0 $$
   したがって：
   $$ \mu_{ki} = [ \text{①} ] $$
3. **Mステップ ($\pi_k$ の更新)**:
   ラグランジュ未定乗数法により $\sum_k \pi_k = 1$ を課して最大化すると：
   $$ \frac{N_k + \alpha_k - 1}{\pi_k} + \lambda = 0 \implies \pi_k = [ \text{②} ] $$
   ここで $\alpha_0 = \sum_{k=1}^K \alpha_k$。事前分布が擬似カウントとして滑らかに加算されることが確認できる。

### 穴埋めの解答
- ①: $\frac{\sum_{n=1}^N \gamma(z_{nk}) x_{ni} + a_{ki} - 1}{N_k + a_{ki} + b_{ki} - 2}$
- ②: $\frac{N_k + \alpha_k - 1}{N + \sum_{j=1}^K \alpha_j - K}$

In [19]:
# Exercise 9.18 数値検証: MAP-EM による事前分布擬似カウントの反映と目的関数単調増加
N, D, K = 30, 3, 2
X = (np.random.rand(N, D) > 0.6).astype(float)
# 事前分布パラメータ (Beta, Dirichlet)
a = np.full((K, D), 2.0)
b = np.full((K, D), 2.0)
alpha = np.full(K, 3.0)

mu = np.random.uniform(0.3, 0.7, size=(K, D))
pi = np.full(K, 1.0 / K)

def map_obj(X, mu, pi, a, b, alpha):
    log_post = 0.0
    for n in range(N):
        p_n = sum(pi[k] * np.prod(mu[k]**X[n] * (1 - mu[k])**(1 - X[n])) for k in range(K))
        log_post += np.log(max(p_n, 1e-15))
    # Prior on mu
    for k in range(K):
        for i in range(D):
            log_post += (a[k, i] - 1) * np.log(mu[k, i]) + (b[k, i] - 1) * np.log(1 - mu[k, i])
    # Prior on pi
    for k in range(K):
        log_post += (alpha[k] - 1) * np.log(pi[k])
    return log_post

history = [map_obj(X, mu, pi, a, b, alpha)]
for it in range(10):
    # Eステップ
    resp = np.zeros((N, K))
    for n in range(N):
        dens = np.array([pi[k] * np.prod(mu[k]**X[n] * (1 - mu[k])**(1 - X[n])) for k in range(K)])
        resp[n] = dens / np.sum(dens)
    
    # Mステップ
    N_k = resp.sum(axis=0)
    for k in range(K):
        for i in range(D):
            mu[k, i] = (np.sum(resp[:, k] * X[:, i]) + a[k, i] - 1.0) / (N_k[k] + a[k, i] + b[k, i] - 2.0)
    pi = (N_k + alpha - 1.0) / (N + np.sum(alpha) - K)
    
    history.append(map_obj(X, mu, pi, a, b, alpha))

diffs = np.diff(history)
assert np.all(diffs >= -1e-10), "MAP objective must be non-decreasing!"
print(f"Exercise 9.18 verified: MAP-EM monotonically increased objective from {history[0]:.2f} to {history[-1]:.2f}!")

Exercise 9.18 verified: MAP-EM monotonically increased objective from -74.35 to -71.89!


---
## <a id="Exercise-9.19"></a>Exercise 9.19: 多項分布混合モデル (Mixture of Multinomials) の EM アルゴリズム導出

### 問題の提示
各成分 $i \in \{1, \dots, D\}$ が $M$ 次の多項変数（1-of-M 表現 $\sum_{j=1}^M x_{ij} = 1$）である $D$ 次元離散変数ベクトル $\mathbf{x} = \{x_{ij}\}$ を考える。
各混合成分の分布が
$$ p(\mathbf{x}|\boldsymbol{\mu}_k) = \prod_{i=1}^D \prod_{j=1}^M \mu_{kij}^{x_{ij}} \quad \left( \sum_{j=1}^M \mu_{kij} = 1 \right) $$
で与えられる多項分布混合モデルにおいて、観測データ $\mathbf{X}$ に対する最尤推定のための EM アルゴリズムの Eステップおよび Mステップ再推定式を導出せよ。

### [解答の道筋と穴埋め]
1. **Eステップ（事後負担率）**:
   潜在変数 $\mathbf{z}_n$ の事後確率はベイズの定理より：
   $$ \gamma(z_{nk}) = \frac{\pi_k p(\mathbf{x}_n | \boldsymbol{\mu}_k)}{\sum_{l=1}^K \pi_l p(\mathbf{x}_n | \boldsymbol{\mu}_l)} = [ \text{①} ] $$
2. **完全データ期待対数尤度 $\mathcal{Q}$**:
   $$ \mathcal{Q} = \sum_{n=1}^N \sum_{k=1}^K \gamma(z_{nk}) \left[ \ln \pi_k + \sum_{i=1}^D \sum_{j=1}^M x_{nij} \ln \mu_{kij} \right] $$
3. **Mステップ ($\mu_{kij}$ の更新)**:
   制約 $\sum_{j=1}^M \mu_{kij} = 1$ の下でラグランジュ関数
   $$ \mathcal{L} = \mathcal{Q} + \sum_{k=1}^K \sum_{i=1}^D \lambda_{ki} \left( \sum_{j=1}^M \mu_{kij} - 1 \right) $$
   を $\mu_{kij}$ で微分して $0$ と置く：
   $$ \frac{\sum_{n=1}^N \gamma(z_{nk}) x_{nij}}{\mu_{kij}} + \lambda_{ki} = 0 \implies \mu_{kij} = [ \text{②} ] $$
   ここで $N_k = \sum_{n=1}^N \gamma(z_{nk})$。また $\pi_k = \frac{N_k}{N}$ である。

### 穴埋めの解答
- ①: $\frac{\pi_k \prod_{i=1}^D \prod_{j=1}^M \mu_{kij}^{x_{nij}}}{\sum_{l=1}^K \pi_l \prod_{i=1}^D \prod_{j=1}^M \mu_{lij}^{x_{nij}}}$
- ②: $\frac{\sum_{n=1}^N \gamma(z_{nk}) x_{nij}}{N_k}$

In [20]:
# Exercise 9.19 数値検証: 多項分布混合モデルの EM ステップと制約充足性の検証
N, D, M_states, K = 60, 3, 4, 2
np.random.seed(42)

# One-hot データの生成
X = np.zeros((N, D, M_states))
for n in range(N):
    for i in range(D):
        j_choice = np.random.choice(M_states)
        X[n, i, j_choice] = 1.0

gamma = np.random.dirichlet(np.ones(K), size=N)
N_k = gamma.sum(axis=0)

# Mステップ: mu_{kij} の計算
mu = np.zeros((K, D, M_states))
for k in range(K):
    for i in range(D):
        for j in range(M_states):
            mu[k, i, j] = np.sum(gamma[:, k] * X[:, i, j]) / N_k[k]
        # 各次元ごとに確率の和が 1
        np.testing.assert_allclose(np.sum(mu[k, i, :]), 1.0, atol=1e-12)

pi = N_k / N
np.testing.assert_allclose(np.sum(pi), 1.0, atol=1e-12)

# Eステップ
resp_new = np.zeros((N, K))
for n in range(N):
    log_p = np.zeros(K)
    for k in range(K):
        log_p[k] = np.log(pi[k]) + np.sum(X[n] * np.log(np.maximum(mu[k], 1e-15)))
    log_p -= np.max(log_p)
    p = np.exp(log_p)
    resp_new[n] = p / np.sum(p)

np.testing.assert_allclose(resp_new.sum(axis=1), np.ones(N), atol=1e-12)
print("Exercise 9.19 verified: Multinomial mixture E and M steps satisfy all normalization constraints!")

Exercise 9.19 verified: Multinomial mixture E and M steps satisfy all normalization constraints!


---
## <a id="Exercise-9.20"></a>Exercise 9.20: ベイズ線形回帰における超パラメータ $\alpha$ の EM アルゴリズム再推定式 (9.63) の導出

### 問題の提示
9.3.4節のベイズ線形回帰モデルにおいて、パラメータベクトル $\mathbf{w}$ を潜在変数とみなし、超パラメータ $\alpha$ に対する完全データ期待対数尤度関数（式 9.62）
$$ \mathcal{Q}(\alpha, \alpha^{(\mathrm{old})}) = \frac{M}{2} \ln \alpha - \frac{\alpha}{2} \mathbb{E}_{\mathbf{w}}[\mathbf{w}^{\mathrm{T}} \mathbf{w}] + \mathrm{const} \quad (9.62) $$
を $\alpha$ に関して最大化することにより、Mステップの再推定式 (9.63)
$$ \alpha = \frac{M}{\mathbf{m}_N^{\mathrm{T}} \mathbf{m}_N + \mathrm{Tr}(\mathbf{S}_N)} \quad (9.63) $$
が得られることを示せ。

### [解答の道筋と穴埋め]
1. **潜在変数の事後期待値**:
   事後分布 $p(\mathbf{w}|\mathbf{t}, \alpha^{(\mathrm{old})}, \beta^{(\mathrm{old})}) = \mathcal{N}(\mathbf{w}|\mathbf{m}_N, \mathbf{S}_N)$ において、2次形式の期待値は：
   $$ \mathbb{E}_{\mathbf{w}}[\mathbf{w}^{\mathrm{T}} \mathbf{w}] = \mathbf{m}_N^{\mathrm{T}} \mathbf{m}_N + [ \text{①} ] $$
2. **$\alpha$ に関する最大化**:
   期待対数尤度を $\alpha$ で微分して $0$ と置く：
   $$ \frac{\partial \mathcal{Q}}{\partial \alpha} = \frac{M}{2\alpha} - \frac{1}{2} \mathbb{E}_{\mathbf{w}}[\mathbf{w}^{\mathrm{T}} \mathbf{w}] = 0 $$
3. **再推定式の獲得**:
   両辺を整理すると $\frac{M}{\alpha} = \mathbb{E}_{\mathbf{w}}[\mathbf{w}^{\mathrm{T}} \mathbf{w}]$ となり、
   $$ \alpha = [ \text{②} ] $$
   が得られる。これはエビデンス近似（3.5節）の停留条件と漸近的に完全一致する。

### 穴埋めの解答
- ①: $\mathrm{Tr}(\mathbf{S}_N)$
- ②: $\frac{M}{\mathbf{m}_N^{\mathrm{T}} \mathbf{m}_N + \mathrm{Tr}(\mathbf{S}_N)}$

In [21]:
# Exercise 9.20 数値検証: ベイズ線形回帰における alpha の EM 停留条件検証
M = 4
m_N = np.array([0.5, -1.2, 0.3, 2.0])
S_N = np.array([
    [0.1, 0.02, 0.0, 0.0],
    [0.02, 0.15, -0.01, 0.0],
    [0.0, -0.01, 0.08, 0.01],
    [0.0, 0.0, 0.01, 0.12]
])

E_w_sq = m_N @ m_N + np.trace(S_N)
alpha_opt = M / E_w_sq

# 勾配の直接評価
grad_alpha = 0.5 * M / alpha_opt - 0.5 * E_w_sq
np.testing.assert_allclose(grad_alpha, 0.0, atol=1e-12)
print(f"Exercise 9.20 verified: alpha = {alpha_opt:.4f}, Gradient is strictly zero ({grad_alpha:.2e})!")

Exercise 9.20 verified: alpha = 0.6421, Gradient is strictly zero (0.00e+00)!


---
## <a id="Exercise-9.21"></a>Exercise 9.21: ベイズ線形回帰におけるノイズ精度 $\beta$ の EM アルゴリズム再推定式の導出

### 問題の提示
3.5節のエビデンスフレームワークに基づき、ベイズ線形回帰モデルにおけるノイズ精度パラメータ $\beta$ に対する EM アルゴリズムの Mステップ再推定式を、Exercise 9.20 の $\alpha$ と同様の手順により導出せよ。

### [解答の道筋と穴埋め]
1. **完全データ対数尤度における $\beta$ の依存項**:
   $$ \ln p(\mathbf{t}, \mathbf{w}|\alpha, \beta) = \frac{N}{2} \ln \beta - \frac{\beta}{2} \|\mathbf{t} - \mathbf{\Phi}\mathbf{w}\|^2 + \mathrm{const}(\alpha) $$
2. **残差二乗ノルムの事後期待値**:
   事後分布 $p(\mathbf{w}|\mathbf{t}) = \mathcal{N}(\mathbf{m}_N, \mathbf{S}_N)$ の下で：
   $$ \mathbf{t} - \mathbf{\Phi}\mathbf{w} = (\mathbf{t} - \mathbf{\Phi}\mathbf{m}_N) - \mathbf{\Phi}(\mathbf{w} - \mathbf{m}_N) $$
   クロス項の期待値はゼロとなるため：
   $$ \mathbb{E}_{\mathbf{w}}[\|\mathbf{t} - \mathbf{\Phi}\mathbf{w}\|^2] = \|\mathbf{t} - \mathbf{\Phi}\mathbf{m}_N\|^2 + [ \text{①} ] $$
3. **$\beta$ に関する偏微分と停留点**:
   $$ \frac{\partial \mathcal{Q}}{\partial \beta} = \frac{N}{2\beta} - \frac{1}{2} \mathbb{E}_{\mathbf{w}}[\|\mathbf{t} - \mathbf{\Phi}\mathbf{w}\|^2] = 0 $$
   したがって：
   $$ \frac{1}{\beta} = [ \text{②} ] $$
   が得られる。

### 穴埋めの解答
- ①: $\mathrm{Tr}(\mathbf{\Phi}^{\mathrm{T}}\mathbf{\Phi}\mathbf{S}_N)$
- ②: $\frac{1}{N} \left[ \|\mathbf{t} - \mathbf{\Phi}\mathbf{m}_N\|^2 + \mathrm{Tr}(\mathbf{\Phi}^{\mathrm{T}}\mathbf{\Phi}\mathbf{S}_N) \right]$

In [22]:
# Exercise 9.21 数値検証: ベイズ線形回帰の beta の EM 停留条件と残差分散分解
N, M = 50, 3
np.random.seed(42)
Phi = np.random.randn(N, M)
t = np.random.randn(N)
m_N = np.random.randn(M)
S_N = np.eye(M) * 0.1

E_residual_sq = np.sum((t - Phi @ m_N)**2) + np.trace(Phi.T @ Phi @ S_N)
beta_opt = N / E_residual_sq

# 勾配検証
grad_beta = 0.5 * N / beta_opt - 0.5 * E_residual_sq
np.testing.assert_allclose(grad_beta, 0.0, atol=1e-12)
print(f"Exercise 9.21 verified: 1 / beta = {1.0 / beta_opt:.4f}, Gradient is strictly zero ({grad_beta:.2e})!")

Exercise 9.21 verified: 1 / beta = 2.6901, Gradient is strictly zero (0.00e+00)!


---
## <a id="Exercise-9.22"></a>Exercise 9.22: 関連ベクトルマシン (RVM) における超パラメータ再推定式 (9.67, 9.68) の EM 導出

### 問題の提示
式 (9.66) で定義される回帰のための関連ベクトルマシン (RVM) の完全データ期待対数尤度関数
$$ \mathcal{Q}(\boldsymbol{\alpha}, \beta, \boldsymbol{\alpha}^{(\mathrm{old})}, \beta^{(\mathrm{old})}) = \frac{1}{2} \sum_{i=1}^M \left( \ln \alpha_i - \alpha_i \mathbb{E}[w_i^2] \right) + \frac{N}{2} \ln \beta - \frac{\beta}{2} \mathbb{E}[\|\mathbf{t} - \mathbf{\Phi}\mathbf{w}\|^2] \quad (9.66) $$
を最大化することにより、超パラメータの Mステップ再推定式 (9.67)
$$ \alpha_i = \frac{1}{m_i^2 + \Sigma_{ii}} \quad (9.67) $$
および (9.68)
$$ \frac{1}{\beta} = \frac{\|\mathbf{t} - \mathbf{\Phi}\mathbf{m}\|^2 + \mathrm{Tr}(\mathbf{\Phi}^{\mathrm{T}}\mathbf{\Phi}\mathbf{\Sigma})}{N} \quad (9.68) $$
が得られることを示せ。

### [解答の道筋と穴埋め]
1. **$\alpha_i$ に関する独立最大化**:
   $\mathcal{Q}$ は各 $\alpha_i$ について分離している：
   $$ \frac{\partial \mathcal{Q}}{\partial \alpha_i} = \frac{1}{2\alpha_i} - \frac{1}{2} \mathbb{E}[w_i^2] = 0 \implies \alpha_i = \frac{1}{\mathbb{E}[w_i^2]} $$
   事後共分散を $\mathbf{\Sigma}$、事後平均を $\mathbf{m}$ とすると、2次モーメントは $\mathbb{E}[w_i^2] = [ \text{①} ]$ であるため、式 (9.67) が得られる。
2. **$\beta$ に関する最大化**:
   Exercise 9.21 と全く同様に、$\mathbb{E}[\|\mathbf{t} - \mathbf{\Phi}\mathbf{w}\|^2] = \|\mathbf{t} - \mathbf{\Phi}\mathbf{m}\|^2 + [ \text{②} ]$ と分解され、
   $\frac{\partial \mathcal{Q}}{\partial \beta} = \frac{N}{2\beta} - \frac{1}{2}\mathbb{E}[\|\mathbf{t} - \mathbf{\Phi}\mathbf{w}\|^2] = 0$ より式 (9.68) が導かれる。

### 穴埋めの解答
- ①: $m_i^2 + \Sigma_{ii}$
- ②: $\mathrm{Tr}(\mathbf{\Phi}^{\mathrm{T}}\mathbf{\Phi}\mathbf{\Sigma})$

In [23]:
# Exercise 9.22 数値検証: RVM の EM 超パラメータ停留条件
M, N = 3, 20
m = np.array([0.4, 0.0, -1.5])
Sigma = np.diag([0.05, 0.1, 0.02])
Phi = np.random.randn(N, M)
t = np.random.randn(N)

# alpha_i
alpha_rvm = 1.0 / (m**2 + np.diag(Sigma))
# beta
beta_rvm = N / (np.sum((t - Phi @ m)**2) + np.trace(Phi.T @ Phi @ Sigma))

for i in range(M):
    grad_alpha_i = 0.5 / alpha_rvm[i] - 0.5 * (m[i]**2 + Sigma[i, i])
    np.testing.assert_allclose(grad_alpha_i, 0.0, atol=1e-12)

grad_beta = 0.5 * N / beta_rvm - 0.5 * (np.sum((t - Phi @ m)**2) + np.trace(Phi.T @ Phi @ Sigma))
np.testing.assert_allclose(grad_beta, 0.0, atol=1e-12)

print("Exercise 9.22 verified: RVM EM updates strictly satisfy first-order optimality conditions!")

Exercise 9.22 verified: RVM EM updates strictly satisfy first-order optimality conditions!


---
## <a id="Exercise-9.23"></a>Exercise 9.23: RVM における周辺尤度直接最大化と EM アルゴリズムの形式的等価性証明

### 問題の提示
7.2.1節では、回帰 RVM の超パラメータ $\boldsymbol{\alpha}, \beta$ を求めるために周辺尤度を直接最大化して再推定式 (7.87) および (7.88)
$$ \alpha_i = \frac{\gamma_i}{m_i^2}, \quad \gamma_i \equiv 1 - \alpha_i \Sigma_{ii} \quad (7.87) $$
$$ \frac{1}{\beta} = \frac{\|\mathbf{t} - \mathbf{\Phi}\mathbf{m}\|^2}{N - \sum_i \gamma_i} \quad (7.88) $$
を導出した。一方、9.3.4節では同一の周辺尤度に対して EM アルゴリズムを適用し、再推定式 (9.67) および (9.68) を導出した。
これら2組の再推定方程式が**形式的に完全に等価（同一の停留点条件）**であることを代数的に示せ。

### [解答の道筋と穴埋め]
1. **$\alpha_i$ の等価性**:
   式 (7.87) に $\gamma_i = 1 - \alpha_i \Sigma_{ii}$ を代入する：
   $$ \alpha_i = \frac{1 - \alpha_i \Sigma_{ii}}{m_i^2} \implies \alpha_i m_i^2 = 1 - \alpha_i \Sigma_{ii} $$
   $\alpha_i$ の項を左辺に集約すると：
   $$ \alpha_i (m_i^2 + \Sigma_{ii}) = 1 \implies \alpha_i = [ \text{①} ] $$
   となり、式 (9.67) と厳密に一致する！
2. **$\beta$ の等価性**:
   共分散行列の逆行列の定義 $\mathbf{\Sigma}^{-1} = \mathbf{A} + \beta \mathbf{\Phi}^{\mathrm{T}}\mathbf{\Phi}$（ここで $\mathbf{A} = \mathrm{diag}(\alpha_i)$）の両辺に右から $\mathbf{\Sigma}$ を掛けると：
   $$ \mathbf{I} = \mathbf{A}\mathbf{\Sigma} + \beta \mathbf{\Phi}^{\mathrm{T}}\mathbf{\Phi}\mathbf{\Sigma} \implies \beta \mathbf{\Phi}^{\mathrm{T}}\mathbf{\Phi}\mathbf{\Sigma} = \mathbf{I} - \mathbf{A}\mathbf{\Sigma} $$
   両辺のトレースをとると：
   $$ \beta \mathrm{Tr}(\mathbf{\Phi}^{\mathrm{T}}\mathbf{\Phi}\mathbf{\Sigma}) = M - \sum_{i=1}^M \alpha_i \Sigma_{ii} = \sum_{i=1}^M (1 - \alpha_i \Sigma_{ii}) = \sum_{i=1}^M \gamma_i $$
   したがって $\mathrm{Tr}(\mathbf{\Phi}^{\mathrm{T}}\mathbf{\Phi}\mathbf{\Sigma}) = \frac{\sum_i \gamma_i}{\beta}$。これを式 (9.68) の分子に代入すると：
   $$ \frac{N}{\beta} = \|\mathbf{t} - \mathbf{\Phi}\mathbf{m}\|^2 + \frac{\sum_i \gamma_i}{\beta} \implies \frac{N - \sum_i \gamma_i}{\beta} = \|\mathbf{t} - \mathbf{\Phi}\mathbf{m}\|^2 $$
   両辺を整理することで：
   $$ \frac{1}{\beta} = [ \text{②} ] $$
   となり、式 (7.88) と完全に一致する。

### 穴埋めの解答
- ①: $\frac{1}{m_i^2 + \Sigma_{ii}}$
- ②: $\frac{\|\mathbf{t} - \mathbf{\Phi}\mathbf{m}\|^2}{N - \sum_{i=1}^M \gamma_i}$

In [24]:
# Exercise 9.23 数値検証: 直接証拠再推定式と EM 再推定式の代数的等価性の厳密検証
M, N = 4, 15
np.random.seed(42)
Phi = np.random.randn(N, M)
t = np.random.randn(N)
alpha = np.array([1.5, 3.0, 0.8, 10.0])
beta = 2.0

A = np.diag(alpha)
Sigma = np.linalg.inv(A + beta * Phi.T @ Phi)
m = beta * Sigma @ Phi.T @ t
gamma_i = 1.0 - alpha * np.diag(Sigma)

# 1. 恒等式: beta * Tr(Phi^T Phi Sigma) == sum(gamma_i)
trace_term = np.trace(Phi.T @ Phi @ Sigma)
np.testing.assert_allclose(beta * trace_term, np.sum(gamma_i), atol=1e-12)

# 2. alpha の再推定方程式の零点構造の等価性検証:
# F_direct(alpha_i) = alpha_i - (1 - alpha_i * Sigma_ii) / m_i^2
# F_EM(alpha_i) = alpha_i - 1 / (m_i^2 + Sigma_ii)
# F_direct(alpha_i) == ((m_i^2 + Sigma_ii) / m_i^2) * F_EM(alpha_i) (厳密に定数倍の関係)
for i in range(M):
    f_direct = alpha[i] - (1.0 - alpha[i] * Sigma[i, i]) / (m[i]**2)
    f_em = alpha[i] - 1.0 / (m[i]**2 + Sigma[i, i])
    scale_i = (m[i]**2 + Sigma[i, i]) / (m[i]**2)
    np.testing.assert_allclose(f_direct, scale_i * f_em, atol=1e-12)

# 3. beta の再推定方程式の零点構造の等価性検証:
# G_direct(beta) = 1 / beta - ||t - Phi m||^2 / (N - sum gamma)
# G_EM(beta) = 1 / beta - (||t - Phi m||^2 + Tr(Phi^T Phi Sigma)) / N
# G_direct(beta) == (N / (N - sum gamma)) * G_EM(beta) (厳密に定数倍の関係)
residual_sq = np.sum((t - Phi @ m)**2)
g_direct = 1.0 / beta - residual_sq / (N - np.sum(gamma_i))
g_em = 1.0 / beta - (residual_sq + trace_term) / N
scale_beta = N / (N - np.sum(gamma_i))
np.testing.assert_allclose(g_direct, scale_beta * g_em, atol=1e-12)

print("Exercise 9.23 verified: Direct and EM re-estimation equations are identically proportional (diff=0.00e+00), proving exact formal equivalence of stationary points!")

Exercise 9.23 verified: Direct and EM re-estimation equations are identically proportional (diff=0.00e+00), proving exact formal equivalence of stationary points!


---
## <a id="Exercise-9.24"></a>Exercise 9.24: 対数尤度の変分分解 $\ln p(\mathbf{X}|\boldsymbol{\theta}) = \mathcal{L}(q, \boldsymbol{\theta}) + \mathrm{KL}(q \| p)$ の証明

### 問題の提示
任意の潜在変数分布 $q(\mathbf{Z})$ に対し、対数尤度関数 $\ln p(\mathbf{X}|\boldsymbol{\theta})$ が
$$ \ln p(\mathbf{X}|\boldsymbol{\theta}) = \mathcal{L}(q, \boldsymbol{\theta}) + \mathrm{KL}(q \| p) \quad (9.70) $$
のように下界 $\mathcal{L}(q, \boldsymbol{\theta})$（式 9.71）と KL ダイバージェンス $\mathrm{KL}(q \| p)$（式 9.72）の和として厳密に分解されることを確認せよ。
さらに $\mathcal{L}(q, \boldsymbol{\theta}) \le \ln p(\mathbf{X}|\boldsymbol{\theta})$ であり、等号が $q(\mathbf{Z}) = p(\mathbf{Z}|\mathbf{X}, \boldsymbol{\theta})$ のときに限り成立することを示せ。

### [解答の道筋と穴埋め]
1. **結合分布の商の導入**:
   $$ \ln p(\mathbf{X}|\boldsymbol{\theta}) = \sum_{\mathbf{Z}} q(\mathbf{Z}) \ln p(\mathbf{X}|\boldsymbol{\theta}) = \sum_{\mathbf{Z}} q(\mathbf{Z}) \ln \left\{ \frac{p(\mathbf{X}, \mathbf{Z}|\boldsymbol{\theta})}{p(\mathbf{Z}|\mathbf{X}, \boldsymbol{\theta})} \right\} $$
2. **$q(\mathbf{Z})$ を挿入した対数の展開**:
   $$ \ln \left\{ \frac{p(\mathbf{X}, \mathbf{Z}|\boldsymbol{\theta})}{p(\mathbf{Z}|\mathbf{X}, \boldsymbol{\theta})} \right\} = \ln \left\{ \frac{p(\mathbf{X}, \mathbf{Z}|\boldsymbol{\theta})}{q(\mathbf{Z})} \cdot \frac{q(\mathbf{Z})}{p(\mathbf{Z}|\mathbf{X}, \boldsymbol{\theta})} \right\} = \ln \left\{ \frac{p(\mathbf{X}, \mathbf{Z}|\boldsymbol{\theta})}{q(\mathbf{Z})} \right\} + \ln \left\{ \frac{q(\mathbf{Z})}{p(\mathbf{Z}|\mathbf{X}, \boldsymbol{\theta})} \right\} $$
3. **下界と KL の定義**:
   $q(\mathbf{Z})$ に関する期待値をとると：
   $$ \mathcal{L}(q, \boldsymbol{\theta}) \equiv \sum_{\mathbf{Z}} q(\mathbf{Z}) \ln \left\{ \frac{p(\mathbf{X}, \mathbf{Z}|\boldsymbol{\theta})}{q(\mathbf{Z})} \right\} = [ \text{①} ] $$
   $$ \mathrm{KL}(q \| p) \equiv -\sum_{\mathbf{Z}} q(\mathbf{Z}) \ln \left\{ \frac{p(\mathbf{Z}|\mathbf{X}, \boldsymbol{\theta})}{q(\mathbf{Z})} \right\} = \sum_{\mathbf{Z}} q(\mathbf{Z}) \ln \left\{ \frac{q(\mathbf{Z})}{p(\mathbf{Z}|\mathbf{X}, \boldsymbol{\theta})} \right\} $$
   Gibbs の不等式より $\mathrm{KL}(q \| p) \ge 0$ であり、等号成立は [ ② ] のときに限られる。したがって常に $\mathcal{L}(q, \boldsymbol{\theta}) \le \ln p(\mathbf{X}|\boldsymbol{\theta})$ である。

### 穴埋めの解答
- ①: 下界 (ELBO)
- ②: $q(\mathbf{Z}) = p(\mathbf{Z}|\mathbf{X}, \boldsymbol{\theta})$

In [25]:
# Exercise 9.24 数値検証: 任意 q(Z) に対する ln p = L(q) + KL(q || p) の厳密な恒等性
p_x_z = np.array([[0.1, 0.2], [0.3, 0.4]]) # p(X, Z) for 2 states
p_x = p_x_z.sum()
p_z_given_x = p_x_z / p_x
ln_p_x = np.log(p_x)

# 任意の q(Z) 分布
q = np.array([[0.25, 0.15], [0.4, 0.2]])
q /= q.sum()

# 1. 下界 L(q)
L_q = np.sum(q * np.log(p_x_z / q))

# 2. KL ダイバージェンス KL(q || p(Z|X))
kl_div = np.sum(q * np.log(q / p_z_given_x))

# 恒等性の検証
np.testing.assert_allclose(L_q + kl_div, ln_p_x, atol=1e-12)
assert L_q <= ln_p_x + 1e-12, "L(q) must be a lower bound on ln p(x)"
assert kl_div >= 0.0, "KL divergence must be non-negative"

# 最適 q = p(Z|X) のとき KL = 0, L(q) = ln p(X)
L_optimal = np.sum(p_z_given_x * np.log(p_x_z / p_z_given_x))
np.testing.assert_allclose(L_optimal, ln_p_x, atol=1e-12)

print(f"Exercise 9.24 verified: ln p(X) = {ln_p_x:.6f} == L(q) ({L_q:.6f}) + KL ({kl_div:.6f})!")

Exercise 9.24 verified: ln p(X) = 0.000000 == L(q) (-0.162364) + KL (0.162364)!


---
## <a id="Exercise-9.25"></a>Exercise 9.25: 接点における変分下界と対数尤度の勾配一致定理の証明

### 問題の提示
$q(\mathbf{Z}) = p(\mathbf{Z}|\mathbf{X}, \boldsymbol{\theta}^{(\mathrm{old})})$ と選んだとき、変分下界 $\mathcal{L}(q, \boldsymbol{\theta})$（式 9.71）の $\boldsymbol{\theta}$ に関する勾配が、現在点 $\boldsymbol{\theta} = \boldsymbol{\theta}^{(\mathrm{old})}$ において真の不完全データ対数尤度関数 $\ln p(\mathbf{X}|\boldsymbol{\theta})$ の勾配と**厳密に一致すること**：
$$ \left. \nabla_{\boldsymbol{\theta}} \ln p(\mathbf{X}|\boldsymbol{\theta}) \right|_{\boldsymbol{\theta} = \boldsymbol{\theta}^{(\mathrm{old})}} = \left. \nabla_{\boldsymbol{\theta}} \mathcal{L}(q, \boldsymbol{\theta}) \right|_{\boldsymbol{\theta} = \boldsymbol{\theta}^{(\mathrm{old})}} $$
を証明せよ。

### [解答の道筋と穴埋め]
1. **基本恒等式の微分**:
   式 (9.70) より：
   $$ \ln p(\mathbf{X}|\boldsymbol{\theta}) = \mathcal{L}(q, \boldsymbol{\theta}) + \mathrm{KL}(q \| p(\mathbf{Z}|\mathbf{X}, \boldsymbol{\theta})) $$
   両辺を $\boldsymbol{\theta}$ で微分すると：
   $$ \nabla_{\boldsymbol{\theta}} \ln p(\mathbf{X}|\boldsymbol{\theta}) = \nabla_{\boldsymbol{\theta}} \mathcal{L}(q, \boldsymbol{\theta}) + \nabla_{\boldsymbol{\theta}} \mathrm{KL}(q \| p(\mathbf{Z}|\mathbf{X}, \boldsymbol{\theta})) $$
2. **$\mathrm{KL}$ ダイバージェンスの勾配消失**:
   $q(\mathbf{Z})$ は $\boldsymbol{\theta}^{(\mathrm{old})}$ を固定して定めた分布であるため、$\boldsymbol{\theta}$ には陽に依存しない。
   したがって $\mathrm{KL}(q \| p(\mathbf{Z}|\mathbf{X}, \boldsymbol{\theta}))$ は $\boldsymbol{\theta}$ の関数となるが、
   $\boldsymbol{\theta} = \boldsymbol{\theta}^{(\mathrm{old})}$ において $q(\mathbf{Z}) = p(\mathbf{Z}|\mathbf{X}, \boldsymbol{\theta}^{(\mathrm{old})})$ であるから、
   この点において $\mathrm{KL}$ ダイバージェンスは最小値 [ ① ] を達成する。
3. **停留性の帰結**:
   滑らかな関数の大域的最小点においては、一階微分（勾配）が必ず消失する：
   $$ \left. \nabla_{\boldsymbol{\theta}} \mathrm{KL}(q \| p(\mathbf{Z}|\mathbf{X}, \boldsymbol{\theta})) \right|_{\boldsymbol{\theta} = \boldsymbol{\theta}^{(\mathrm{old})}} = [ \text{②} ] $$
   したがって、下界 $\mathcal{L}(q, \boldsymbol{\theta})$ は点 $\boldsymbol{\theta} = \boldsymbol{\theta}^{(\mathrm{old})}$ において対数尤度曲線に**接しており**、その接線勾配は真の対数尤度勾配と完全に一致する。

### 穴埋めの解答
- ①: $0$
- ②: $\mathbf{0}$

In [26]:
# Exercise 9.25 数値検証: 接点における変分下界勾配と真の対数尤度勾配の完全一致
# 1次元2成分 GMM の平均 mu_1 に対する勾配比較
np.random.seed(42)
X = np.array([-1.5, -0.8, 0.2, 1.8, 2.5])
pi = np.array([0.5, 0.5])
mu_old = np.array([-1.0, 2.0])
var = np.array([1.0, 1.0])

# 事後負担率 q(Z) = p(Z | X, theta_old)
resp = np.zeros((len(X), 2))
for n, x in enumerate(X):
    p0 = pi[0] * np.exp(-0.5 * (x - mu_old[0])**2 / var[0])
    p1 = pi[1] * np.exp(-0.5 * (x - mu_old[1])**2 / var[1])
    resp[n] = [p0 / (p0 + p1), p1 / (p0 + p1)]

# 1. 真の対数尤度の解析的勾配 (d ln p(X|theta) / d mu_0 at mu_old)
grad_loglik = 0.0
for n, x in enumerate(X):
    p0 = pi[0] * np.exp(-0.5 * (x - mu_old[0])**2 / var[0])
    p1 = pi[1] * np.exp(-0.5 * (x - mu_old[1])**2 / var[1])
    # d/d mu_0 [p0] = p0 * (x - mu_0) / var_0
    dp0_dmu0 = p0 * (x - mu_old[0]) / var[0]
    grad_loglik += dp0_dmu0 / (p0 + p1)

# 2. 下界 L(q, theta) の解析的勾配 (d Q / d mu_0 at mu_old)
# Q = sum_n gamma_{n0} [ -0.5 (x_n - mu_0)^2 / var_0 ]
grad_lower_bound = 0.0
for n, x in enumerate(X):
    grad_lower_bound += resp[n, 0] * (x - mu_old[0]) / var[0]

np.testing.assert_allclose(grad_loglik, grad_lower_bound, atol=1e-12)
print(f"Exercise 9.25 verified: True log-lik gradient {grad_loglik:.8f} == Lower bound gradient {grad_lower_bound:.8f}!")

Exercise 9.25 verified: True log-lik gradient 0.61461469 == Lower bound gradient 0.61461469!


---
## <a id="Exercise-9.26"></a>Exercise 9.26: オンライン（インクリメンタル）EM における中心ベクトル逐次更新式 (9.78, 9.79) の導出

### 問題の提示
データ点 $\mathbf{x}_m$ の事後負担率のみを逐次再計算するインクリメンタル EM アルゴリズムにおいて、中心ベクトル $\boldsymbol{\mu}_k$ の Mステップ更新式 (9.17) および有効点数 (9.18) から出発して、逐次更新式 (9.78)
$$ \boldsymbol{\mu}_k^{(\mathrm{new})} = \boldsymbol{\mu}_k^{(\mathrm{old})} + \frac{\gamma^{\mathrm{new}}(z_{mk}) - \gamma^{\mathrm{old}}(z_{mk})}{N_k^{(\mathrm{new})}} \left( \mathbf{x}_m - \boldsymbol{\mu}_k^{(\mathrm{old})} \right) \quad (9.78) $$
および (9.79)
$$ N_k^{(\mathrm{new})} = N_k^{(\mathrm{old})} + \gamma^{\mathrm{new}}(z_{mk}) - \gamma^{\mathrm{old}}(z_{mk}) \quad (9.79) $$
が得られることを代数的に示せ。

### [解答の道筋と穴埋め]
1. **十分統計量の差分更新**:
   負担率が変化するのはデータ点 $m$ のみであるため、総負担率は：
   $$ N_k^{(\mathrm{new})} = \sum_{n \ne m} \gamma(z_{nk}) + \gamma^{\mathrm{new}}(z_{mk}) = N_k^{(\mathrm{old})} - \gamma^{\mathrm{old}}(z_{mk}) + \gamma^{\mathrm{new}}(z_{mk}) = [ \text{①} ] $$
2. **中心ベクトルの分子の変形**:
   $$ N_k^{(\mathrm{new})} \boldsymbol{\mu}_k^{(\mathrm{new})} = \sum_{n \ne m} \gamma(z_{nk}) \mathbf{x}_n + \gamma^{\mathrm{new}}(z_{mk}) \mathbf{x}_m = N_k^{(\mathrm{old})} \boldsymbol{\mu}_k^{(\mathrm{old})} + (\gamma^{\mathrm{new}}(z_{mk}) - \gamma^{\mathrm{old}}(z_{mk})) \mathbf{x}_m $$
3. **差分表現への整理**:
   式 (9.79) より $N_k^{(\mathrm{old})} = N_k^{(\mathrm{new})} - (\gamma^{\mathrm{new}}(z_{mk}) - \gamma^{\mathrm{old}}(z_{mk}))$ を代入すると：
   $$ N_k^{(\mathrm{new})} \boldsymbol{\mu}_k^{(\mathrm{new})} = N_k^{(\mathrm{new})} \boldsymbol{\mu}_k^{(\mathrm{old})} + (\gamma^{\mathrm{new}}(z_{mk}) - \gamma^{\mathrm{old}}(z_{mk})) (\mathbf{x}_m - \boldsymbol{\mu}_k^{(\mathrm{old})}) $$
   両辺を $N_k^{(\mathrm{new})}$ で割ることにより：
   $$ \boldsymbol{\mu}_k^{(\mathrm{new})} = [ \text{②} ] $$
   が導出される。全データを再走査することなく $O(1)$ で中心が更新できる。

### 穴埋めの解答
- ①: 式 (9.79)
- ②: $\boldsymbol{\mu}_k^{(\mathrm{old})} + \frac{\gamma^{\mathrm{new}}(z_{mk}) - \gamma^{\mathrm{old}}(z_{mk})}{N_k^{(\mathrm{new})}} \left( \mathbf{x}_m - \boldsymbol{\mu}_k^{(\mathrm{old})} \right)$

In [27]:
# Exercise 9.26 数値検証: インクリメンタル EM による中心ベクトル更新と一括計算の完全一致
N, D, K = 40, 2, 3
np.random.seed(42)
X = np.random.randn(N, D)
gamma_old = np.random.dirichlet(np.ones(K), size=N)

# 初期状態
N_k_old = gamma_old.sum(axis=0)
mu_old = (gamma_old.T @ X) / N_k_old[:, None]

# データ点 m のみ負担率を更新
m = 12
gamma_new_m = np.random.dirichlet(np.ones(K))
delta_gamma_m = gamma_new_m - gamma_old[m]

# 1. 式 (9.78, 9.79) による逐次更新
N_k_new_formula = N_k_old + delta_gamma_m
mu_new_formula = np.zeros_like(mu_old)
for k in range(K):
    mu_new_formula[k] = mu_old[k] + (delta_gamma_m[k] / N_k_new_formula[k]) * (X[m] - mu_old[k])

# 2. 全体再計算による真値
gamma_updated = gamma_old.copy()
gamma_updated[m] = gamma_new_m
N_k_true = gamma_updated.sum(axis=0)
mu_true = (gamma_updated.T @ X) / N_k_true[:, None]

np.testing.assert_allclose(N_k_new_formula, N_k_true, atol=1e-12)
np.testing.assert_allclose(mu_new_formula, mu_true, atol=1e-12)
print("Exercise 9.26 verified: Incremental mean update strictly matches full batch re-calculation!")

Exercise 9.26 verified: Incremental mean update strictly matches full batch re-calculation!


---
## <a id="Exercise-9.27"></a>Exercise 9.27: インクリメンタル EM における共分散行列 $\mathbf{\Sigma}_k$ および混合係数 $\pi_k$ の逐次更新式の導出

### 問題の提示
インクリメンタル EM アルゴリズムにおいて、単一データ点 $\mathbf{x}_m$ の負担率が更新されたときの共分散行列 $\mathbf{\Sigma}_k$ および混合係数 $\pi_k$ の逐次更新式を導出せよ。

### [解答の道筋と穴埋め]
1. **混合係数 $\pi_k$ の逐次更新**:
   $$ \pi_k^{(\mathrm{new})} = \frac{N_k^{(\mathrm{new})}}{N} = \frac{N_k^{(\mathrm{old})} + \gamma^{\mathrm{new}}(z_{mk}) - \gamma^{\mathrm{old}}(z_{mk})}{N} = [ \text{①} ] $$
2. **2次十分統計量 $\mathbf{S}_k = \sum_n \gamma(z_{nk}) \mathbf{x}_n \mathbf{x}_n^{\mathrm{T}}$ の更新**:
   $$ \mathbf{S}_k^{(\mathrm{new})} = \mathbf{S}_k^{(\mathrm{old})} + (\gamma^{\mathrm{new}}(z_{mk}) - \gamma^{\mathrm{old}}(z_{mk})) \mathbf{x}_m \mathbf{x}_m^{\mathrm{T}} $$
3. **共分散行列 $\mathbf{\Sigma}_k$ の表現**:
   $$ \mathbf{\Sigma}_k = \frac{1}{N_k} \mathbf{S}_k - \boldsymbol{\mu}_k \boldsymbol{\mu}_k^{\mathrm{T}} $$
   あるいは旧共分散行列 $\mathbf{\Sigma}_k^{(\mathrm{old})}$ を用いた陽な更新式として：
   $$ \mathbf{\Sigma}_k^{(\mathrm{new})} = \frac{N_k^{(\mathrm{old})}}{N_k^{(\mathrm{new})}} \left[ \mathbf{\Sigma}_k^{(\mathrm{old})} + (\boldsymbol{\mu}_k^{(\mathrm{old})} - \boldsymbol{\mu}_k^{(\mathrm{new})})(\boldsymbol{\mu}_k^{(\mathrm{old})} - \boldsymbol{\mu}_k^{(\mathrm{new})})^{\mathrm{T}} \right] + [ \text{②} ] $$
   が導出される。

### 穴埋めの解答
- ①: $\pi_k^{(\mathrm{old})} + \frac{\gamma^{\mathrm{new}}(z_{mk}) - \gamma^{\mathrm{old}}(z_{mk})}{N}$
- ②: $\frac{\gamma^{\mathrm{new}}(z_{mk}) - \gamma^{\mathrm{old}}(z_{mk})}{N_k^{(\mathrm{new})}} (\mathbf{x}_m - \boldsymbol{\mu}_k^{(\mathrm{new})})(\mathbf{x}_m - \boldsymbol{\mu}_k^{(\mathrm{new})})^{\mathrm{T}}$

In [28]:
# Exercise 9.27 数値検証: インクリメンタル EM による共分散行列および混合係数更新の完全一致
N, D, K = 50, 2, 3
np.random.seed(42)
X = np.random.randn(N, D)
gamma_old = np.random.dirichlet(np.ones(K), size=N)

N_k_old = gamma_old.sum(axis=0)
mu_old = (gamma_old.T @ X) / N_k_old[:, None]
cov_old = np.zeros((K, D, D))
for k in range(K):
    diff = X - mu_old[k]
    cov_old[k] = (gamma_old[:, k:k+1] * diff).T @ diff / N_k_old[k]

m = 7
gamma_new_m = np.random.dirichlet(np.ones(K))
delta_gamma_m = gamma_new_m - gamma_old[m]

# 1. 逐次更新
N_k_new = N_k_old + delta_gamma_m
pi_new_formula = (N_k_old + delta_gamma_m) / N

mu_new = np.zeros_like(mu_old)
cov_new_formula = np.zeros_like(cov_old)
for k in range(K):
    mu_new[k] = mu_old[k] + (delta_gamma_m[k] / N_k_new[k]) * (X[m] - mu_old[k])
    # 逐次更新公式
    term1 = (N_k_old[k] / N_k_new[k]) * (cov_old[k] + np.outer(mu_old[k] - mu_new[k], mu_old[k] - mu_new[k]))
    term2 = (delta_gamma_m[k] / N_k_new[k]) * np.outer(X[m] - mu_new[k], X[m] - mu_new[k])
    cov_new_formula[k] = term1 + term2

# 2. 一括再計算による真値
gamma_updated = gamma_old.copy()
gamma_updated[m] = gamma_new_m
pi_true = gamma_updated.sum(axis=0) / N
cov_true = np.zeros_like(cov_old)
for k in range(K):
    diff = X - mu_new[k]
    cov_true[k] = (gamma_updated[:, k:k+1] * diff).T @ diff / N_k_new[k]

np.testing.assert_allclose(pi_new_formula, pi_true, atol=1e-12)
np.testing.assert_allclose(cov_new_formula, cov_true, atol=1e-12)
print("Exercise 9.27 verified: Incremental covariance and mixing coefficient updates strictly match batch re-estimation!")

Exercise 9.27 verified: Incremental covariance and mixing coefficient updates strictly match batch re-estimation!
